In [1]:
import csv
import math
import random
import igraph
import datetime
import numpy as np
import pandas as pd
from igraph import *
from sklearn import metrics
from collections import Counter
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('mode.chained_assignment', None)
import warnings 
warnings.filterwarnings('ignore')

# 标签为2019-2023被引数量，作为公司的排名
# 把2019年之前的数据，进行分析。  TopN N = 20 100 200 500 1000

In [2]:
edges_1 = pd.read_excel("./results/最终数据/train_data.xlsx",sheet_name=0)
edges_2 = pd.read_excel("./results/最终数据/train_data.xlsx",sheet_name=1)

In [3]:
test_df = pd.read_excel("./results/最终数据/test.xlsx")

# P_count算法
Tips: P_count算法是指将专利数量越多的专利权人，影响力越大

In [4]:
group_data = edges_2.groupby('cited').count()
group_data = group_data.reset_index()
group_data

,cited,citing,Assignee_type,Assignee_score,order,year
0,12西格玛控股有限公司,1,1,1,1,1
1,3M创新有限公司,13,13,13,13,13
2,3WIN股份有限公司,1,1,1,1,1
3,3形状股份有限公司,2,2,2,2,2
4,460医药公司,1,1,1,1,1
...,...,...,...,...,...,...
10205,龙迅半导体（合肥）股份有限公司,1,1,1,1,1
10206,龙马智芯（珠海横琴）科技有限公司,2,2,2,2,2
10207,龟田俊忠,1,1,1,1,1
10208,（株）赛丽康,1,1,1,1,1


In [5]:
p_count_merge = pd.merge(test_df,group_data,on='cited')
p_count_merge['P_count_rank'] = p_count_merge['citing'].rank(axis=0,method='min',ascending=False)
p_count_merge

,cited,citation_count,rank,citing,Assignee_type,Assignee_score,order,year,P_count_rank
0,西安电子科技大学,778,1,884,884,884,884,884,3.0
1,电子科技大学,637,2,582,582,582,582,582,5.0
2,华南理工大学,604,3,263,263,263,263,263,18.0
3,华中科技大学,547,4,300,300,300,300,300,13.0
4,浙江大学,547,4,406,406,406,406,406,10.0
...,...,...,...,...,...,...,...,...,...
4055,麻省理工学院,1,3783,1,1,1,1,1,2386.0
4056,金色熊猫有限公司,1,3783,2,2,2,2,2,1742.0
4057,中科水滴科技（深圳）有限公司,1,3783,1,1,1,1,1,2386.0
4058,吴蕾,1,3783,1,1,1,1,1,2386.0


In [11]:
p_count_merge.to_excel("./results/算法结果/P_count_rank.xlsx",index=None)

In [6]:
N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
y_list = []
for top_N in N_list:
    sub_data = p_count_merge.iloc[:top_N]['P_count_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)  
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg)
    y_list.append(len(in_data))
# print('AUC score:',metrics.auc(N_list, y_list))

Top N count:  10 , 总数量：  7 ,Average： 0.7 , NDCG@K score:  0.732
Top N count:  20 , 总数量：  15 ,Average： 0.75 , NDCG@K score:  0.828
Top N count:  30 , 总数量：  22 ,Average： 0.733 , NDCG@K score:  0.809
Top N count:  40 , 总数量：  29 ,Average： 0.725 , NDCG@K score:  0.802
Top N count:  50 , 总数量：  34 ,Average： 0.68 , NDCG@K score:  0.761
Top N count:  60 , 总数量：  40 ,Average： 0.667 , NDCG@K score:  0.754
Top N count:  70 , 总数量：  46 ,Average： 0.657 , NDCG@K score:  0.742
Top N count:  80 , 总数量：  50 ,Average： 0.625 , NDCG@K score:  0.714
Top N count:  90 , 总数量：  54 ,Average： 0.6 , NDCG@K score:  0.694
Top N count:  100 , 总数量：  64 ,Average： 0.64 , NDCG@K score:  0.723
Top N count:  200 , 总数量：  131 ,Average： 0.655 , NDCG@K score:  0.72
Top N count:  300 , 总数量：  196 ,Average： 0.653 , NDCG@K score:  0.71
Top N count:  400 , 总数量：  263 ,Average： 0.657 , NDCG@K score:  0.71
Top N count:  500 , 总数量：  320 ,Average： 0.64 , NDCG@K score:  0.691
Top N count:  1000 , 总数量：  677 ,Average： 0.677 , NDCG@K score:  0

# Citation_Count算法
Tips: Citation_Count算法是指专利权人的引证次数之和

In [7]:
new_df = pd.DataFrame()
new_df['patent'] = edges_1.groupby("cited").count().index
new_df['citation'] = edges_1.groupby('cited').count()['citing'].values
edges_copy = edges_2.copy()
edges_copy.columns = ['patent','cited','Assignee_type','Assignee_score','order','year']
merge_data = pd.merge(edges_copy, new_df, on='patent')
merge_data

,patent,cited,Assignee_type,Assignee_score,order,year,citation
0,CN106127195B,OPPO广东移动通信有限公司,E,2,1,2017,4
1,CN106384105B,南昌欧菲生物识别技术有限公司,E,2,1,2019,2
2,CN106330576B,北京大麦文化传媒发展有限公司,E,2,1,2019,1
3,CN107483260B,北京三快在线科技有限公司,E,2,1,2021,1
4,CN106934320B,小米科技有限责任公司,E,2,1,2020,4
...,...,...,...,...,...,...,...
28688,CN107316011B,杭州励飞软件技术有限公司,E,2,1,2021,1
28689,CN111094095B,动态AD有限责任公司,E,2,1,2021,1
28690,CN107818311B,豪威科技（北京）股份有限公司,E,2,1,2021,1
28691,CN108471946B,珀迪迈垂克斯公司,E,2,1,2023,1


In [10]:
sum_data = merge_data.groupby("cited").sum()
sum_data = sum_data.reset_index()
citation_data = sum_data[['cited','citation']]
citation_data

,cited,citation
0,12西格玛控股有限公司,7
1,3M创新有限公司,13
2,3WIN股份有限公司,2
3,3形状股份有限公司,2
4,A.D.整体应用有限公司,1
...,...,...
6323,龙芯中科技术股份有限公司,5
6324,龙迅半导体（合肥）股份有限公司,1
6325,龙马智芯（珠海横琴）科技有限公司,1
6326,龟田俊忠,2


In [11]:
cc_merge = pd.merge(test_df,citation_data,on='cited')
cc_merge['citation_count_rank'] = cc_merge['citation'].rank(axis=0,method='min',ascending=False)
cc_merge

,cited,citation_count,rank,citation,citation_count_rank
0,西安电子科技大学,778,1,1483,3.0
1,电子科技大学,637,2,894,4.0
2,华南理工大学,604,3,483,11.0
3,华中科技大学,547,4,451,13.0
4,浙江大学,547,4,598,9.0
...,...,...,...,...,...
2894,北京中科信利技术有限公司,1,3783,6,940.0
2895,天地伟业技术有限公司,1,3783,3,1407.0
2896,江苏杰瑞科技集团有限责任公司,1,3783,2,1703.0
2897,麻省理工学院,1,3783,1,2129.0


In [17]:
cc_merge.to_excel("./results/算法结果/Citation_count_rank.xlsx",index=None)

In [12]:
N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = cc_merge.iloc[:top_N]['citation_count_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg)  
    
# citation_count_merge.iloc[:200].to_csv("Top_200_citation_count.csv",index=None)

Top N count:  10 , 总数量：  6 ,Average： 0.6 , NDCG@K score:  0.653
Top N count:  20 , 总数量：  15 ,Average： 0.75 , NDCG@K score:  0.822
Top N count:  30 , 总数量：  21 ,Average： 0.7 , NDCG@K score:  0.783
Top N count:  40 , 总数量：  27 ,Average： 0.675 , NDCG@K score:  0.764
Top N count:  50 , 总数量：  32 ,Average： 0.64 , NDCG@K score:  0.732
Top N count:  60 , 总数量：  38 ,Average： 0.633 , NDCG@K score:  0.722
Top N count:  70 , 总数量：  42 ,Average： 0.6 , NDCG@K score:  0.695
Top N count:  80 , 总数量：  50 ,Average： 0.625 , NDCG@K score:  0.709
Top N count:  90 , 总数量：  57 ,Average： 0.633 , NDCG@K score:  0.71
Top N count:  100 , 总数量：  63 ,Average： 0.63 , NDCG@K score:  0.706
Top N count:  200 , 总数量：  126 ,Average： 0.63 , NDCG@K score:  0.698
Top N count:  300 , 总数量：  189 ,Average： 0.63 , NDCG@K score:  0.689
Top N count:  400 , 总数量：  249 ,Average： 0.623 , NDCG@K score:  0.678
Top N count:  500 , 总数量：  313 ,Average： 0.626 , NDCG@K score:  0.678
Top N count:  1000 , 总数量：  640 ,Average： 0.64 , NDCG@K score:  0.6

# Citation_Count_W算法
Tips: Citation_Count_W算法是指专利的被引次数，再按照专利权人进行平均得到的引证次数之和

In [13]:
new_df = edges_2.copy()
pa = new_df.groupby('citing').count()
pa = pa.reset_index()
pa
pa_df = pa[['citing','year']]
pa_df.columns = ['patent','node_count']

In [14]:
cd_merge = pd.merge(merge_data, pa_df, on='patent')
cd_merge

,patent,cited,Assignee_type,Assignee_score,order,year,citation,node_count
0,CN106127195B,OPPO广东移动通信有限公司,E,2,1,2017,4,1
1,CN106384105B,南昌欧菲生物识别技术有限公司,E,2,1,2019,2,1
2,CN106330576B,北京大麦文化传媒发展有限公司,E,2,1,2019,1,1
3,CN107483260B,北京三快在线科技有限公司,E,2,1,2021,1,1
4,CN106934320B,小米科技有限责任公司,E,2,1,2020,4,1
...,...,...,...,...,...,...,...,...
28688,CN107316011B,杭州励飞软件技术有限公司,E,2,1,2021,1,1
28689,CN111094095B,动态AD有限责任公司,E,2,1,2021,1,1
28690,CN107818311B,豪威科技（北京）股份有限公司,E,2,1,2021,1,1
28691,CN108471946B,珀迪迈垂克斯公司,E,2,1,2023,1,1


In [15]:
cd_merge['citation_count_w'] = cd_merge['citation'] / cd_merge['node_count']
cd_merge

,patent,cited,Assignee_type,Assignee_score,order,year,citation,node_count,citation_count_w
0,CN106127195B,OPPO广东移动通信有限公司,E,2,1,2017,4,1,4.0
1,CN106384105B,南昌欧菲生物识别技术有限公司,E,2,1,2019,2,1,2.0
2,CN106330576B,北京大麦文化传媒发展有限公司,E,2,1,2019,1,1,1.0
3,CN107483260B,北京三快在线科技有限公司,E,2,1,2021,1,1,1.0
4,CN106934320B,小米科技有限责任公司,E,2,1,2020,4,1,4.0
...,...,...,...,...,...,...,...,...,...
28688,CN107316011B,杭州励飞软件技术有限公司,E,2,1,2021,1,1,1.0
28689,CN111094095B,动态AD有限责任公司,E,2,1,2021,1,1,1.0
28690,CN107818311B,豪威科技（北京）股份有限公司,E,2,1,2021,1,1,1.0
28691,CN108471946B,珀迪迈垂克斯公司,E,2,1,2023,1,1,1.0


In [16]:
sum_ccw = cd_merge.groupby('cited').sum()
sum_ccw = sum_ccw.reset_index()
ccw_data = sum_ccw[['cited','citation_count_w']]
ccw_data

,cited,citation_count_w
0,12西格玛控股有限公司,7.0
1,3M创新有限公司,13.0
2,3WIN股份有限公司,2.0
3,3形状股份有限公司,2.0
4,A.D.整体应用有限公司,1.0
...,...,...
6323,龙芯中科技术股份有限公司,5.0
6324,龙迅半导体（合肥）股份有限公司,1.0
6325,龙马智芯（珠海横琴）科技有限公司,1.0
6326,龟田俊忠,2.0


In [17]:
ccw_merge = pd.merge(test_df,ccw_data,on='cited')
ccw_merge['CCW_rank'] = ccw_merge['citation_count_w'].rank(axis=0,method='min',ascending=False)
# ccw_merge.to_excel("./results/算法结果/Citation_count_weight_rank.xlsx",index=None)

In [18]:
N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = ccw_merge.iloc[:top_N]['CCW_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg)  
# ccw_merge.iloc[:200].to_csv("Top_200_ccw.csv",index=None)

Top N count:  10 , 总数量：  6 ,Average： 0.6 , NDCG@K score:  0.653
Top N count:  20 , 总数量：  16 ,Average： 0.8 , NDCG@K score:  0.861
Top N count:  30 , 总数量：  22 ,Average： 0.733 , NDCG@K score:  0.812
Top N count:  40 , 总数量：  26 ,Average： 0.65 , NDCG@K score:  0.746
Top N count:  50 , 总数量：  31 ,Average： 0.62 , NDCG@K score:  0.714
Top N count:  60 , 总数量：  38 ,Average： 0.633 , NDCG@K score:  0.722
Top N count:  70 , 总数量：  44 ,Average： 0.629 , NDCG@K score:  0.717
Top N count:  80 , 总数量：  50 ,Average： 0.625 , NDCG@K score:  0.709
Top N count:  90 , 总数量：  58 ,Average： 0.644 , NDCG@K score:  0.721
Top N count:  100 , 总数量：  64 ,Average： 0.64 , NDCG@K score:  0.715
Top N count:  200 , 总数量：  128 ,Average： 0.64 , NDCG@K score:  0.706
Top N count:  300 , 总数量：  188 ,Average： 0.627 , NDCG@K score:  0.687
Top N count:  400 , 总数量：  245 ,Average： 0.613 , NDCG@K score:  0.669
Top N count:  500 , 总数量：  302 ,Average： 0.604 , NDCG@K score:  0.659
Top N count:  1000 , 总数量：  626 ,Average： 0.626 , NDCG@K score:

# H-index算法

In [19]:
H_list = dict()
for i in merge_data.groupby('cited'):
    Assignee,sub_data = i[0],i[1]
    sub_data['rank'] = sub_data['citation'].rank(method='min',ascending=False)
    sub_data.sort_values(by=['citation'],ascending=[False],ignore_index=True,inplace=True)
    sub_data = sub_data.reset_index(drop=True)
    
    h = 0
    for index in sub_data.index:
        row_data = sub_data.iloc[index]
        citation, rank = row_data['citation'],row_data['rank']
        if rank + 1 <= citation:
            h += 1
    H_list[Assignee] = h

In [20]:
H_df = pd.DataFrame()
H_df['cited'] = H_list.keys()
H_df['H_index'] = H_list.values()
H_merge = pd.merge(test_df,H_df,on='cited')
H_merge['H_index_rank'] = H_merge['H_index'].rank(axis=0,method='min',ascending=False)
# H_merge.to_excel("./results/算法结果/H_index_rank.xlsx",index=None)

In [21]:
N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = H_merge.iloc[:top_N]['H_index_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg)  


Top N count:  10 , 总数量：  5 ,Average： 0.5 , NDCG@K score:  0.383
Top N count:  20 , 总数量：  11 ,Average： 0.55 , NDCG@K score:  0.642
Top N count:  30 , 总数量：  21 ,Average： 0.7 , NDCG@K score:  0.781
Top N count:  40 , 总数量：  23 ,Average： 0.575 , NDCG@K score:  0.68
Top N count:  50 , 总数量：  30 ,Average： 0.6 , NDCG@K score:  0.691
Top N count:  60 , 总数量：  32 ,Average： 0.533 , NDCG@K score:  0.633
Top N count:  70 , 总数量：  40 ,Average： 0.571 , NDCG@K score:  0.654
Top N count:  80 , 总数量：  45 ,Average： 0.562 , NDCG@K score:  0.64
Top N count:  90 , 总数量：  47 ,Average： 0.522 , NDCG@K score:  0.605
Top N count:  100 , 总数量：  71 ,Average： 0.71 , NDCG@K score:  0.768
Top N count:  200 , 总数量：  142 ,Average： 0.71 , NDCG@K score:  0.76
Top N count:  300 , 总数量：  174 ,Average： 0.58 , NDCG@K score:  0.644
Top N count:  400 , 总数量：  285 ,Average： 0.713 , NDCG@K score:  0.752
Top N count:  500 , 总数量：  326 ,Average： 0.652 , NDCG@K score:  0.697
Top N count:  1000 , 总数量：  804 ,Average： 0.804 , NDCG@K score:  0.8

# G-inde算法

In [22]:
G_list = dict()
for i in merge_data.groupby("cited"):
    Assignee,sub_data = i[0],i[1]
    sum_citation = 0
    sub_data['rank'] =sub_data['citation'].rank(method='min', ascending=False)
    sub_data.sort_values(by=['citation'],ascending=[False],ignore_index=True,inplace=True)
    sub_data = sub_data.reset_index(drop=True)
    g_index = 0
    for index in sub_data.index:
        row_data = sub_data.iloc[index]
        citation, rank = row_data['citation'],row_data['rank']
        sum_citation += citation
        g_square = math.pow(index+1,2)
        if g_square <= sum_citation:
            g_index += 1
    G_list[Assignee] = g_index

In [23]:
G_df = pd.DataFrame()
G_df['cited'] = G_list.keys()
G_df['G_index'] = G_list.values()

G_index_merge = pd.merge(test_df,G_df,on='cited')
G_index_merge['G_index_rank'] = G_index_merge['G_index'].rank(axis=0,method='min',ascending=False)
G_index_merge
# G_index_merge.to_csv("G_index_rank.csv",index=None)

,cited,citation_count,rank,G_index,G_index_rank
0,西安电子科技大学,778,1,10,10.0
1,电子科技大学,637,2,11,6.0
2,华南理工大学,604,3,11,6.0
3,华中科技大学,547,4,9,17.0
4,浙江大学,547,4,11,6.0
...,...,...,...,...,...
2894,北京中科信利技术有限公司,1,3783,2,523.0
2895,天地伟业技术有限公司,1,3783,1,1075.0
2896,江苏杰瑞科技集团有限责任公司,1,3783,1,1075.0
2897,麻省理工学院,1,3783,1,1075.0


In [30]:
 G_index_merge.to_excel("./results/算法结果/G_index_rank.xlsx",index=None)

In [24]:
N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500]
for top_N in N_list:
    sub_data = G_index_merge.iloc[:top_N]['G_index_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 
# G_index_merge.iloc[:200].to_csv("Top_200_g_index.csv",index=None)

Top N count:  10 , 总数量：  7 ,Average： 0.7 , NDCG@K score:  0.763
Top N count:  20 , 总数量：  14 ,Average： 0.7 , NDCG@K score:  0.762
Top N count:  30 , 总数量：  18 ,Average： 0.6 , NDCG@K score:  0.688
Top N count:  40 , 总数量：  26 ,Average： 0.65 , NDCG@K score:  0.742
Top N count:  50 , 总数量：  29 ,Average： 0.58 , NDCG@K score:  0.68
Top N count:  60 , 总数量：  31 ,Average： 0.517 , NDCG@K score:  0.624
Top N count:  70 , 总数量：  46 ,Average： 0.657 , NDCG@K score:  0.729
Top N count:  80 , 总数量：  50 ,Average： 0.625 , NDCG@K score:  0.7
Top N count:  90 , 总数量：  55 ,Average： 0.611 , NDCG@K score:  0.684
Top N count:  100 , 总数量：  74 ,Average： 0.74 , NDCG@K score:  0.794
Top N count:  200 , 总数量：  135 ,Average： 0.675 , NDCG@K score:  0.731
Top N count:  300 , 总数量：  220 ,Average： 0.733 , NDCG@K score:  0.774
Top N count:  400 , 总数量：  261 ,Average： 0.652 , NDCG@K score:  0.701
Top N count:  500 , 总数量：  292 ,Average： 0.584 , NDCG@K score:  0.638


# SPR算法
Tips: SPR的方法是PN专利引证网络中，将得到的PR的值加总得到专利权人的影响力，不按专利人数平均

In [25]:
citing = edges_1['citing'].tolist()
cited = edges_1['cited'].tolist()
edges = []
for p in zip(citing,cited):
    edges.append((p[0],p[1]))

g = Graph.TupleList(edges,vertex_name_attr='name',edge_attrs=None,directed=True)

In [26]:
pr_df = pd.DataFrame([{"citing":p[0]['name'], 'PR':p[1]} for p in zip(g.vs, g.pagerank(damping=0.5))])
new_df = edges_2.copy()
pa = new_df.groupby('citing').count()
pa = pa.reset_index()
pa_df = pa[['citing','year']]
pa_df.columns = ['citing','node_count']
pa_df

,citing,node_count
0,CN100334460C,1
1,CN100334837C,1
2,CN100334838C,1
3,CN100334841C,1
4,CN100335002C,1
...,...,...
46530,CN1996879B,1
46531,CN1997903B,1
46532,CN1997905B,1
46533,CN1997906B,1


In [27]:
pr_merge = pd.merge(new_df, pa_df, on='citing')
pr_merge

,citing,cited,Assignee_type,Assignee_score,order,year,node_count
0,CN105844614B,广东工业大学,U,3,1,2019,1
1,CN104608125B,精工爱普生株式会社,E,2,1,2019,1
2,CN106239505B,广东电网有限责任公司电力科学研究院,R,3,1,2018,1
3,CN108972536B,西门子（中国）有限公司,E,2,1,2021,1
4,CN109154276B,维斯塔斯风力系统集团公司,E,2,1,2020,1
...,...,...,...,...,...,...,...
51192,CN107316011B,杭州励飞软件技术有限公司,E,2,1,2021,1
51193,CN111094095B,动态AD有限责任公司,E,2,1,2021,1
51194,CN107818311B,豪威科技（北京）股份有限公司,E,2,1,2021,1
51195,CN108471946B,珀迪迈垂克斯公司,E,2,1,2023,1


In [28]:
pr_merge_1 = pd.merge(pr_merge, pr_df, on='citing')
pr_merge_1

,citing,cited,Assignee_type,Assignee_score,order,year,node_count,PR
0,CN105844614B,广东工业大学,U,3,1,2019,1,0.000015
1,CN104608125B,精工爱普生株式会社,E,2,1,2019,1,0.000015
2,CN106239505B,广东电网有限责任公司电力科学研究院,R,3,1,2018,1,0.000015
3,CN108972536B,西门子（中国）有限公司,E,2,1,2021,1,0.000015
4,CN109154276B,维斯塔斯风力系统集团公司,E,2,1,2020,1,0.000015
...,...,...,...,...,...,...,...,...
51192,CN107316011B,杭州励飞软件技术有限公司,E,2,1,2021,1,0.000018
51193,CN111094095B,动态AD有限责任公司,E,2,1,2021,1,0.000016
51194,CN107818311B,豪威科技（北京）股份有限公司,E,2,1,2021,1,0.000019
51195,CN108471946B,珀迪迈垂克斯公司,E,2,1,2023,1,0.000017


In [29]:
sum_pr = pr_merge_1.groupby("cited").sum()
sum_pr = sum_pr.reset_index()
pr_data = sum_pr[['cited','PR']]
pr_data

,cited,PR
0,12西格玛控股有限公司,0.000039
1,3M创新有限公司,0.000228
2,3WIN股份有限公司,0.000020
3,3形状股份有限公司,0.000041
4,460医药公司,0.000015
...,...,...
10205,龙迅半导体（合肥）股份有限公司,0.000023
10206,龙马智芯（珠海横琴）科技有限公司,0.000033
10207,龟田俊忠,0.000026
10208,（株）赛丽康,0.000023


In [30]:
SPR_merge = pd.merge(test_df,pr_data,on='cited')
SPR_merge['SPR_rank'] = SPR_merge['PR'].rank(axis=0, method='min', ascending=False)
SPR_merge

,cited,citation_count,rank,PR,SPR_rank
0,西安电子科技大学,778,1,0.021858,3.0
1,电子科技大学,637,2,0.013832,5.0
2,华南理工大学,604,3,0.006527,16.0
3,华中科技大学,547,4,0.007150,13.0
4,浙江大学,547,4,0.009717,9.0
...,...,...,...,...,...
4055,麻省理工学院,1,3783,0.000019,2996.0
4056,金色熊猫有限公司,1,3783,0.000030,2372.0
4057,中科水滴科技（深圳）有限公司,1,3783,0.000015,3252.0
4058,吴蕾,1,3783,0.000018,3137.0


In [38]:
SPR_merge.to_excel("./results/算法结果/SPR_rank.xlsx",index=None)

In [31]:
N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = SPR_merge.iloc[:top_N]['SPR_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 
# SPR_merge.iloc[:200].to_csv("Top_200_spr.csv",index=None)

Top N count:  10 , 总数量：  7 ,Average： 0.7 , NDCG@K score:  0.732
Top N count:  20 , 总数量：  15 ,Average： 0.75 , NDCG@K score:  0.828
Top N count:  30 , 总数量：  22 ,Average： 0.733 , NDCG@K score:  0.809
Top N count:  40 , 总数量：  29 ,Average： 0.725 , NDCG@K score:  0.802
Top N count:  50 , 总数量：  35 ,Average： 0.7 , NDCG@K score:  0.775
Top N count:  60 , 总数量：  39 ,Average： 0.65 , NDCG@K score:  0.734
Top N count:  70 , 总数量：  45 ,Average： 0.643 , NDCG@K score:  0.729
Top N count:  80 , 总数量：  46 ,Average： 0.575 , NDCG@K score:  0.672
Top N count:  90 , 总数量：  55 ,Average： 0.611 , NDCG@K score:  0.7
Top N count:  100 , 总数量：  63 ,Average： 0.63 , NDCG@K score:  0.711
Top N count:  200 , 总数量：  128 ,Average： 0.64 , NDCG@K score:  0.707
Top N count:  300 , 总数量：  190 ,Average： 0.633 , NDCG@K score:  0.694
Top N count:  400 , 总数量：  253 ,Average： 0.632 , NDCG@K score:  0.687
Top N count:  500 , 总数量：  314 ,Average： 0.628 , NDCG@K score:  0.681
Top N count:  1000 , 总数量：  633 ,Average： 0.633 , NDCG@K score:  

# SPR_W算法
Tips: SPR_W算法是指按照专利引证网络计算得到的PageRank得分后平分给专利权人（数量）

In [32]:
pr_merge_1['spr_w'] = pr_merge_1['PR']/pr_merge_1['node_count']
sum_prw = pr_merge_1.groupby("cited").sum()
sum_prw = sum_prw.reset_index()
prw_data = sum_prw[['cited','spr_w']]
prw_data

,cited,spr_w
0,12西格玛控股有限公司,0.000039
1,3M创新有限公司,0.000228
2,3WIN股份有限公司,0.000020
3,3形状股份有限公司,0.000041
4,460医药公司,0.000008
...,...,...
10205,龙迅半导体（合肥）股份有限公司,0.000023
10206,龙马智芯（珠海横琴）科技有限公司,0.000033
10207,龟田俊忠,0.000026
10208,（株）赛丽康,0.000023


In [33]:
SPRW_merge = pd.merge(test_df,prw_data,on='cited')
SPRW_merge['SPRw_rank'] = SPRW_merge['spr_w'].rank(axis=0,method='min',ascending=False)
# SPRW_merge.to_excel("./results/算法结果/SPRw_rank.xlsx",index=None)
# SPRW_merge.to_csv("SPRW_rank.csv",index=None)

In [34]:
N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = SPRW_merge.iloc[:top_N]['SPRw_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 
# SPRW_merge.iloc[:200].to_csv("Top_200_sprw.csv",index=None)

Top N count:  10 , 总数量：  7 ,Average： 0.7 , NDCG@K score:  0.732
Top N count:  20 , 总数量：  16 ,Average： 0.8 , NDCG@K score:  0.861
Top N count:  30 , 总数量：  22 ,Average： 0.733 , NDCG@K score:  0.811
Top N count:  40 , 总数量：  29 ,Average： 0.725 , NDCG@K score:  0.802
Top N count:  50 , 总数量：  36 ,Average： 0.72 , NDCG@K score:  0.79
Top N count:  60 , 总数量：  40 ,Average： 0.667 , NDCG@K score:  0.751
Top N count:  70 , 总数量：  45 ,Average： 0.643 , NDCG@K score:  0.729
Top N count:  80 , 总数量：  50 ,Average： 0.625 , NDCG@K score:  0.715
Top N count:  90 , 总数量：  57 ,Average： 0.633 , NDCG@K score:  0.716
Top N count:  100 , 总数量：  64 ,Average： 0.64 , NDCG@K score:  0.721
Top N count:  200 , 总数量：  127 ,Average： 0.635 , NDCG@K score:  0.703
Top N count:  300 , 总数量：  194 ,Average： 0.647 , NDCG@K score:  0.705
Top N count:  400 , 总数量：  256 ,Average： 0.64 , NDCG@K score:  0.695
Top N count:  500 , 总数量：  309 ,Average： 0.618 , NDCG@K score:  0.672
Top N count:  1000 , 总数量：  625 ,Average： 0.625 , NDCG@K score:

# AN_AuthorNetwork 专利权人引证的结果 （不删除自引）
Tips: AN算法是指根据专利的引证情况，构建专利权人的引证网络，然后采用PageRank算法进行分析专利权人引证网络中的影响力

In [35]:
an_data = pd.read_excel("./results/最终数据/AN_data.xlsx")
an_data

,citing,citing_assignees,cited,cited_assignees
0,CN105844614B,广东工业大学,CN105260710B,北京石油化工学院
1,CN104608125B,精工爱普生株式会社,CN105234938B,精工爱普生株式会社
2,CN106239505B,广东电网有限责任公司电力科学研究院,CN105234938B,精工爱普生株式会社
3,CN108972536B,西门子（中国）有限公司,CN105234938B,精工爱普生株式会社
4,CN109154276B,维斯塔斯风力系统集团公司,CN105257475B,北京科诺伟业科技股份有限公司;科诺伟业风能设备（北京）有限公司
...,...,...,...,...
54350,CN111727108B,欧姆龙株式会社,CN110325327B,西门子股份公司;加利福尼亚大学董事会
54351,CN109464125B,四川大学华西第二医院,CN108471946B,珀迪迈垂克斯公司
54352,CN109859159B,浙江大学,CN110073404B,南坦生物组学有限责任公司
54353,CN110355755B,深圳铭杰医疗科技有限公司,CN108942918B,沈阳建筑大学


In [36]:
data = pd.DataFrame()
data['cited_a'] = an_data['cited_assignees']
data['citing_a'] = an_data['citing_assignees']

# 作者引用网络关系(不删除自引用的数据)
cited_citing = []
for index in data.index:
    cited_a,citing_a = data.iloc[index]['cited_a'], data.iloc[index]['citing_a']
    for i in cited_a.split(";"):
        for j in citing_a.split(";"):
            cited_citing.append((i,j))

In [37]:
from collections import Counter
cc_dict = dict(Counter(cited_citing))
pd_df = pd.DataFrame.from_dict(cc_dict,orient='index',columns=['count']).reset_index()

edges_a = pd_df['index'].tolist()
edges_w = pd_df['count'].tolist()

assignee_g = Graph.TupleList(edges_a,vertex_name_attr='name',edge_attrs=None,directed=True)
an_pr = pd.DataFrame([{"cited":p[0]['name'], 'AN_PR':p[1]} for p in zip(assignee_g.vs, assignee_g.pagerank(damping=0.5))])

an_pr_merge = pd.merge(test_df,an_pr,on='cited')
an_pr_merge['AN_PR_rank'] = an_pr_merge['AN_PR'].rank(axis=0,method='min',ascending=False)
an_pr_merge
# an_pr_merge.to_csv("AN_PR_rank.csv",index=None)

,cited,citation_count,rank,AN_PR,AN_PR_rank
0,西安电子科技大学,778,1,0.001737,11.0
1,电子科技大学,637,2,0.002740,4.0
2,华南理工大学,604,3,0.001662,12.0
3,华中科技大学,547,4,0.001375,15.0
4,浙江大学,547,4,0.001545,13.0
...,...,...,...,...,...
4055,麻省理工学院,1,3783,0.000065,3416.0
4056,金色熊猫有限公司,1,3783,0.000100,1345.0
4057,中科水滴科技（深圳）有限公司,1,3783,0.000089,1662.0
4058,吴蕾,1,3783,0.000071,2728.0


In [46]:
an_pr_merge.to_excel("./results/算法结果/AN_PR_rank.xlsx",index=None)

In [38]:
N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = an_pr_merge.iloc[:top_N]['AN_PR_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 
# an_pr_merge.iloc[:200].to_csv("Top_200_anpr.csv",index=None)

Top N count:  10 , 总数量：  4 ,Average： 0.4 , NDCG@K score:  0.36
Top N count:  20 , 总数量：  11 ,Average： 0.55 , NDCG@K score:  0.678
Top N count:  30 , 总数量：  18 ,Average： 0.6 , NDCG@K score:  0.71
Top N count:  40 , 总数量：  25 ,Average： 0.625 , NDCG@K score:  0.72
Top N count:  50 , 总数量：  34 ,Average： 0.68 , NDCG@K score:  0.754
Top N count:  60 , 总数量：  42 ,Average： 0.7 , NDCG@K score:  0.77
Top N count:  70 , 总数量：  47 ,Average： 0.671 , NDCG@K score:  0.749
Top N count:  80 , 总数量：  52 ,Average： 0.65 , NDCG@K score:  0.733
Top N count:  90 , 总数量：  57 ,Average： 0.633 , NDCG@K score:  0.72
Top N count:  100 , 总数量：  62 ,Average： 0.62 , NDCG@K score:  0.705
Top N count:  200 , 总数量：  118 ,Average： 0.59 , NDCG@K score:  0.665
Top N count:  300 , 总数量：  190 ,Average： 0.633 , NDCG@K score:  0.692
Top N count:  400 , 总数量：  240 ,Average： 0.6 , NDCG@K score:  0.661
Top N count:  500 , 总数量：  286 ,Average： 0.572 , NDCG@K score:  0.632
Top N count:  1000 , 总数量：  571 ,Average： 0.571 , NDCG@K score:  0.618
To

# AN_带权重边的算法结果

In [39]:
an_prw = pd.DataFrame([{"cited":p[0]['name'], 'AN_PRW':p[1]} for p in zip(assignee_g.vs, assignee_g.pagerank(damping=0.5,weights=edges_w))])

an_prw_merge = pd.merge(test_df,an_prw,on='cited')
an_prw_merge['AN_PRW_rank'] = an_prw_merge['AN_PRW'].rank(axis=0,method='min',ascending=False)
# an_prw_merge.to_excel("./results/算法结果/AN_PRW_rank.xlsx",index=None)

In [40]:
N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = an_prw_merge.iloc[:top_N]['AN_PRW_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 

Top N count:  10 , 总数量：  5 ,Average： 0.5 , NDCG@K score:  0.58
Top N count:  20 , 总数量：  9 ,Average： 0.45 , NDCG@K score:  0.58
Top N count:  30 , 总数量：  18 ,Average： 0.6 , NDCG@K score:  0.71
Top N count:  40 , 总数量：  24 ,Average： 0.6 , NDCG@K score:  0.701
Top N count:  50 , 总数量：  35 ,Average： 0.7 , NDCG@K score:  0.768
Top N count:  60 , 总数量：  41 ,Average： 0.683 , NDCG@K score:  0.758
Top N count:  70 , 总数量：  47 ,Average： 0.671 , NDCG@K score:  0.748
Top N count:  80 , 总数量：  52 ,Average： 0.65 , NDCG@K score:  0.733
Top N count:  90 , 总数量：  55 ,Average： 0.611 , NDCG@K score:  0.701
Top N count:  100 , 总数量：  61 ,Average： 0.61 , NDCG@K score:  0.698
Top N count:  200 , 总数量：  120 ,Average： 0.6 , NDCG@K score:  0.674
Top N count:  300 , 总数量：  187 ,Average： 0.623 , NDCG@K score:  0.685
Top N count:  400 , 总数量：  234 ,Average： 0.585 , NDCG@K score:  0.648
Top N count:  500 , 总数量：  289 ,Average： 0.578 , NDCG@K score:  0.637
Top N count:  1000 , 总数量：  566 ,Average： 0.566 , NDCG@K score:  0.614
T

# AN_no_self 不包含专利权人自引
Tips: AN_no_self算法是指专利权人引证数据中，去除专利权人自引部分的结果

In [41]:
# 作者引用网络关系(不删除自引用的数据)
cited_citing = []
for index in data.index:
    cited_a,citing_a = data.iloc[index]['cited_a'], data.iloc[index]['citing_a']
    for i in cited_a.split(";"):
        for j in citing_a.split(";"):
            if i != j:
                cited_citing.append((i,j))

In [42]:
from collections import Counter
cc_dict = dict(Counter(cited_citing))
pd_df = pd.DataFrame.from_dict(cc_dict,orient='index',columns=['count']).reset_index()
edges_a = pd_df['index'].tolist()
edges_w = pd_df['count'].tolist()

assignee_g = Graph.TupleList(edges_a,vertex_name_attr='name',edge_attrs=None,directed=True)
an_pr = pd.DataFrame([{"cited":p[0]['name'], 'AN_PR(no_self)':p[1]} for p in zip(assignee_g.vs, assignee_g.pagerank(damping=0.5))])

an_pr_merge = pd.merge(test_df,an_pr,on='cited')
an_pr_merge['AN_PR(no_self)_rank'] = an_pr_merge['AN_PR(no_self)'].rank(axis=0,method='min',ascending=False)

# an_pr_merge.to_csv("AN_PR(no_self)_rank.csv",index=None)
an_pr_merge.to_excel("./results/算法结果/AN_PR(no_self)_rank.xlsx",index=None)
# 评价
N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = an_pr_merge.iloc[:top_N]['AN_PR(no_self)_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 
# an_pr_merge.iloc[:200].to_csv("Top_200_anpr_noself.csv",index=None)

Top N count:  10 , 总数量：  4 ,Average： 0.4 , NDCG@K score:  0.36
Top N count:  20 , 总数量：  12 ,Average： 0.6 , NDCG@K score:  0.712
Top N count:  30 , 总数量：  19 ,Average： 0.633 , NDCG@K score:  0.734
Top N count:  40 , 总数量：  26 ,Average： 0.65 , NDCG@K score:  0.739
Top N count:  50 , 总数量：  35 ,Average： 0.7 , NDCG@K score:  0.768
Top N count:  60 , 总数量：  42 ,Average： 0.7 , NDCG@K score:  0.77
Top N count:  70 , 总数量：  48 ,Average： 0.686 , NDCG@K score:  0.761
Top N count:  80 , 总数量：  52 ,Average： 0.65 , NDCG@K score:  0.733
Top N count:  90 , 总数量：  56 ,Average： 0.622 , NDCG@K score:  0.71
Top N count:  100 , 总数量：  64 ,Average： 0.64 , NDCG@K score:  0.721
Top N count:  200 , 总数量：  121 ,Average： 0.605 , NDCG@K score:  0.678
Top N count:  300 , 总数量：  191 ,Average： 0.637 , NDCG@K score:  0.695
Top N count:  400 , 总数量：  244 ,Average： 0.61 , NDCG@K score:  0.67
Top N count:  500 , 总数量：  287 ,Average： 0.574 , NDCG@K score:  0.635
Top N count:  1000 , 总数量：  573 ,Average： 0.573 , NDCG@K score:  0.62
T

In [52]:
# AN_no_self带权重边的算法结果

In [43]:
an_prw = pd.DataFrame([{"cited":p[0]['name'], 'AN_PRW(no_self)':p[1]} for p in zip(assignee_g.vs, assignee_g.pagerank(damping=0.5,weights=edges_w))])

an_prw_merge = pd.merge(test_df,an_prw,on='cited')
an_prw_merge['AN_PRW(no_self)_rank'] = an_prw_merge['AN_PRW(no_self)'].rank(axis=0,method='min',ascending=False)

an_prw_merge.to_excel("./results/算法结果/AN_PRW(no_self)_rank.xlsx",index=None)

N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = an_prw_merge.iloc[:top_N]['AN_PRW(no_self)_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 
# an_prw_merge.iloc[:200].to_csv("Top_200_anprw_noself.csv",index=None)

Top N count:  10 , 总数量：  4 ,Average： 0.4 , NDCG@K score:  0.36
Top N count:  20 , 总数量：  11 ,Average： 0.55 , NDCG@K score:  0.678
Top N count:  30 , 总数量：  18 ,Average： 0.6 , NDCG@K score:  0.705
Top N count:  40 , 总数量：  25 ,Average： 0.625 , NDCG@K score:  0.721
Top N count:  50 , 总数量：  35 ,Average： 0.7 , NDCG@K score:  0.768
Top N count:  60 , 总数量：  42 ,Average： 0.7 , NDCG@K score:  0.77
Top N count:  70 , 总数量：  47 ,Average： 0.671 , NDCG@K score:  0.747
Top N count:  80 , 总数量：  53 ,Average： 0.662 , NDCG@K score:  0.744
Top N count:  90 , 总数量：  57 ,Average： 0.633 , NDCG@K score:  0.72
Top N count:  100 , 总数量：  62 ,Average： 0.62 , NDCG@K score:  0.705
Top N count:  200 , 总数量：  118 ,Average： 0.59 , NDCG@K score:  0.666
Top N count:  300 , 总数量：  189 ,Average： 0.63 , NDCG@K score:  0.691
Top N count:  400 , 总数量：  244 ,Average： 0.61 , NDCG@K score:  0.67
Top N count:  500 , 总数量：  286 ,Average： 0.572 , NDCG@K score:  0.633
Top N count:  1000 , 总数量：  574 ,Average： 0.574 , NDCG@K score:  0.622
T

# RW算法_随机游走算法
Tips: RW算法是以专利-专利权人网络构建，并用随机游走算法来衡量专利权人的影响力

In [44]:
edges = [(p[0], p[1]) for p in zip(edges_2['citing'], edges_2['cited'])]  # 构建网络的边数据

In [45]:
h_g = Graph.TupleList(edges, vertex_name_attr='name', edge_attrs=None, directed=True)

In [46]:
assignees_nodes = []
assignees_names = []
patents_nodes = []
patents_names = []
for i in h_g.vs:
    if 'CN' == i['name'][:2]:
        patents_names.append(i['name'])
        patents_nodes.append(i.index)
    else:
        assignees_nodes.append(i.index)
        assignees_names.append(i['name'])

In [47]:
patents_df = pd.DataFrame(zip(patents_names,patents_nodes),columns=['Patent','nodes'])
assignees_df = pd.DataFrame(zip(assignees_names,assignees_nodes), columns=['Assignee','nodes'])

N_a, N_p, d = len(assignees_nodes), len(patents_nodes), 0.5
W_ca = {i:[j.source for j in h_g.vs[i].all_edges()] for i in assignees_nodes}
W_cp = {i:[j.target for j in h_g.vs[i].all_edges()] for i in patents_nodes}

In [48]:
W_p = {i:1/N_p for i in patents_nodes}
W_a = {i:1/N_a for i in assignees_nodes}
W_a0, W_p0 = W_a.copy(), W_p.copy()

for i in range(10000):  
    diff = 0 # 初始化偏差
    W_a_1 = W_a.copy()
    W_p_1 = W_p.copy()
    
    # 遍历专利权人
    for key in list(W_a.keys()):
        tmp = 0
        for paper in W_ca[key]:
            tmp += W_p_1[paper]/len(W_cp[paper]) 
        W_a[key] = (1 - d)/N_a + d * tmp 
        diff += abs(W_a[key] - W_a_1[key])
    
        
    # 专利
    for key in list(W_p.keys()):  # 遍历专利数据
        tmp = 0
        for author in W_cp[key]:
            tmp += W_a_1[author]/len(W_ca[author])
        W_p[key] = (1 - d)/N_p + d * tmp
        diff += abs(W_p[key] - W_p_1[key])
    
    
    print(i, 'diff: ', diff)
    if diff < 0.00000001:
        break


0 diff:  1.2664496178044473
1 diff:  0.6119715182899985
2 diff:  0.2982115040358392
3 diff:  0.14704339335981714
4 diff:  0.07280212238233884
5 diff:  0.03618046625152611
6 diff:  0.018026591315262856
7 diff:  0.008988032177242491
8 diff:  0.004483837091354662
9 diff:  0.002237882498188886
10 diff:  0.0011172125613678379
11 diff:  0.0005578288045655832
12 diff:  0.00027856029615865865
13 diff:  0.0001391153434985344
14 diff:  6.947998545159877e-05
15 diff:  3.4703879428521755e-05
16 diff:  1.733455442610702e-05
17 diff:  8.659083737710814e-06
18 diff:  4.325584618497777e-06
19 diff:  2.160928427249871e-06
20 diff:  1.079564579033446e-06
21 diff:  5.393576956078874e-07
22 diff:  2.6947797085861363e-07
23 diff:  1.346432983541433e-07
24 diff:  6.727743745039322e-08
25 diff:  3.361912530087352e-08
26 diff:  1.680193398284506e-08
27 diff:  8.397636703763356e-09


In [49]:
rw = pd.DataFrame.from_dict(W_a,orient='index',columns=['RW']).reset_index()
rw.columns = ['nodes','RW']
result_merge = pd.merge(assignees_df,rw,on='nodes')
result_merge.columns = ['cited','nodes','RW']

In [50]:
rw_merge = pd.merge(test_df,result_merge,on='cited')
rw_merge['RW_rank'] = rw_merge['RW'].rank(axis=0,method='min',ascending=False)
rw_merge.to_excel("./results/算法结果/RW_rank.xlsx",index=None)

In [51]:
# 
N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = rw_merge.iloc[:top_N]['RW_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 

# rw_merge.iloc[:200].to_csv("Top_200_rw.csv",index=None)

Top N count:  10 , 总数量：  7 ,Average： 0.7 , NDCG@K score:  0.732
Top N count:  20 , 总数量：  15 ,Average： 0.75 , NDCG@K score:  0.828
Top N count:  30 , 总数量：  22 ,Average： 0.733 , NDCG@K score:  0.809
Top N count:  40 , 总数量：  29 ,Average： 0.725 , NDCG@K score:  0.802
Top N count:  50 , 总数量：  36 ,Average： 0.72 , NDCG@K score:  0.79
Top N count:  60 , 总数量：  41 ,Average： 0.683 , NDCG@K score:  0.767
Top N count:  70 , 总数量：  48 ,Average： 0.686 , NDCG@K score:  0.764
Top N count:  80 , 总数量：  52 ,Average： 0.65 , NDCG@K score:  0.734
Top N count:  90 , 总数量：  58 ,Average： 0.644 , NDCG@K score:  0.729
Top N count:  100 , 总数量：  63 ,Average： 0.63 , NDCG@K score:  0.715
Top N count:  200 , 总数量：  131 ,Average： 0.655 , NDCG@K score:  0.721
Top N count:  300 , 总数量：  193 ,Average： 0.643 , NDCG@K score:  0.701
Top N count:  400 , 总数量：  262 ,Average： 0.655 , NDCG@K score:  0.707
Top N count:  500 , 总数量：  317 ,Average： 0.634 , NDCG@K score:  0.687
Top N count:  1000 , 总数量：  640 ,Average： 0.64 , NDCG@K score:

# PRW算法_将PageRank得到的结果加入到RW模型中
Tips: 将PageRank算法加入

In [52]:
citing = edges_1['citing'].tolist()
cited = edges_1['cited'].tolist()
edges = []
for p in zip(citing,cited):
    edges.append((p[0],p[1]))
g = Graph.TupleList(edges,vertex_name_attr='name',edge_attrs=None,directed=True)
pr_1 = [{"Patent":p[0]['name'], 'PR':p[1]} for p in zip(g.vs, g.pagerank(damping=0.5))]
vertices_df = pd.DataFrame(pr_1) 

In [53]:
# 将论文重要性的分数带入到专利-专利权人网络中
patents_merge = pd.merge(vertices_df,patents_df,on='Patent')
patents_merge

,Patent,PR,nodes
0,CN105844614B,0.000015,0
1,CN105260710B,0.000019,39427
2,CN104608125B,0.000015,2
3,CN105234938B,0.000024,39428
4,CN106239505B,0.000015,4
...,...,...,...
46530,CN111094095B,0.000016,56739
46531,CN107818311B,0.000019,56740
46532,CN109979593B,0.000015,39426
46533,CN108471946B,0.000017,56741


In [54]:
N_a, N_p, d = len(assignees_nodes), len(patents_nodes), 0.5
W_p = dict(zip(patents_merge['nodes'],patents_merge['PR']))
W_a = {i:1/N_a for i in assignees_nodes}

W_ca = {}
for i in assignees_nodes:    
    W_ca[i] = [j.source for j in h_g.vs[i].all_edges()]
W_cp = {}
for i in patents_nodes:
    W_cp[i] = [j.target for j in h_g.vs[i].all_edges()]
    
W_p0,W_a0 = W_p.copy(), W_a.copy()

for i in range(10000):  
    diff = 0 # 初始化偏差
    W_a_1 = W_a.copy()
    W_p_1 = W_p.copy()
    for key in list(W_a.keys()):
        tmp = 0
        for paper in W_ca[key]:
            tmp += W_p_1[paper]/len(W_cp[paper]) 
        W_a[key] = (1 - d)*W_a0[key] + d * tmp
        diff += abs(W_a[key] - W_a_1[key])
        
   
    for key in list(W_p.keys()):  # 遍历专利数据
        tmp = 0
        for author in W_cp[key]:
            tmp += W_a_1[author]/len(W_ca[author])
        W_p[key] = (1 - d)*W_p0[key] + d * tmp
        diff += abs(W_p[key] - W_p_1[key])
    
    print(i, 'diff: ', diff)
    if diff < 0.00000001:
        break

0 diff:  1.3085433066463983
1 diff:  0.6307839813911907
2 diff:  0.30759584189708866
3 diff:  0.15171130983627387
4 diff:  0.07515017132329656
5 diff:  0.03735602379933167
6 diff:  0.018609037235560854
7 diff:  0.009277919241576041
8 diff:  0.004628322657532115
9 diff:  0.0023098734706775074
10 diff:  0.0011530764385355857
11 diff:  0.0005756950271875745
12 diff:  0.000287462960161502
13 diff:  0.00014355204701268473
14 diff:  7.169108948780936e-05
15 diff:  3.580550306931405e-05
16 diff:  1.788370369910925e-05
17 diff:  8.932834503550421e-06
18 diff:  4.462056984486034e-06
19 diff:  2.228980851304408e-06
20 diff:  1.11350972693746e-06
21 diff:  5.562902645853501e-07
22 diff:  2.779230147154076e-07
23 diff:  1.388572884350409e-07
24 diff:  6.938004239611151e-08
25 diff:  3.466923148531742e-08
26 diff:  1.7326432763554447e-08
27 diff:  8.65957456349006e-09


In [55]:
prw = pd.DataFrame.from_dict(W_a,orient='index',columns=['PRW']).reset_index()

prw.columns = ['nodes','RW']
result_merge = pd.merge(assignees_df,prw,on='nodes')
result_merge.columns = ['cited','nodes','PRW']
prw_merge = pd.merge(test_df,result_merge,on='cited')

prw_merge['PRW_rank'] = prw_merge['PRW'].rank(axis=0,method='min',ascending=False)
prw_merge.to_excel("./results/算法结果/PRW_rank.xlsx",index=None)

In [56]:
# 
N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = prw_merge.iloc[:top_N]['PRW_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 
# prw_merge.iloc[:200].to_csv("Top_200_prw.csv",index=None)

Top N count:  10 , 总数量：  7 ,Average： 0.7 , NDCG@K score:  0.732
Top N count:  20 , 总数量：  16 ,Average： 0.8 , NDCG@K score:  0.861
Top N count:  30 , 总数量：  22 ,Average： 0.733 , NDCG@K score:  0.809
Top N count:  40 , 总数量：  29 ,Average： 0.725 , NDCG@K score:  0.802
Top N count:  50 , 总数量：  36 ,Average： 0.72 , NDCG@K score:  0.79
Top N count:  60 , 总数量：  40 ,Average： 0.667 , NDCG@K score:  0.751
Top N count:  70 , 总数量：  45 ,Average： 0.643 , NDCG@K score:  0.729
Top N count:  80 , 总数量：  51 ,Average： 0.637 , NDCG@K score:  0.724
Top N count:  90 , 总数量：  58 ,Average： 0.644 , NDCG@K score:  0.726
Top N count:  100 , 总数量：  65 ,Average： 0.65 , NDCG@K score:  0.729
Top N count:  200 , 总数量：  129 ,Average： 0.645 , NDCG@K score:  0.711
Top N count:  300 , 总数量：  196 ,Average： 0.653 , NDCG@K score:  0.71
Top N count:  400 , 总数量：  259 ,Average： 0.647 , NDCG@K score:  0.701
Top N count:  500 , 总数量：  312 ,Average： 0.624 , NDCG@K score:  0.677
Top N count:  1000 , 总数量：  622 ,Average： 0.622 , NDCG@K score:

# TAPRW_基于时间感知的学者学术影响力评估算法

In [58]:
patent_age = pd.read_excel("./results/最终数据/HIN_node.xlsx")
patent_age.columns = ['patent','granted_date','age','AI_keywords_count','AI_COUNT']
patent_age

,patent,granted_date,age,AI_keywords_count,AI_COUNT
0,CN105844614B,2019-08-23 00:00:00,4,0,0.0
1,CN112214368B,2022-06-14 00:00:00,1,0,0.0
2,CN112910720B,2021-08-03 00:00:00,2,0,0.0
3,CN104608125B,2019-12-17 00:00:00,4,0,0.0
4,CN106239505B,2018-12-04 00:00:00,5,0,0.0
...,...,...,...,...,...
74412,CN112454353B,2022-04-15 00:00:00,1,0,0.0
74413,CN110139730B,2022-09-16 00:00:00,1,0,0.0
74414,CN108471946B,2023-03-17 00:00:00,0,0,0.0
74415,CN110073404B,2023-03-21 00:00:00,0,0,0.0


In [59]:
citing = edges_1['citing'].tolist()
cited = edges_1['cited'].tolist()
edges = []
for p in zip(citing,cited):
    edges.append((p[0],p[1]))
g = Graph.TupleList(edges,vertex_name_attr='name',edge_attrs=None,directed=True)

# 计算所有专利时间权重的加和
# patent_age['pre_age'] = 1/(patent_age['age']+1)  # 时间函数f(age_i) 采用倒数形式来表示
patent_age['pre_age'] = [math.exp(-0.1* i) for i in patent_age['age'].tolist()] # 采用 指数时间函数

id_mapping = dict((k['name'],k.index) for k in g.vs)
g_vertice = pd.DataFrame.from_dict(id_mapping,orient='index',columns=['id']).reset_index()
g_vertice.columns=['patent','id']
g_vertice

,patent,id
0,CN105844614B,0
1,CN105260710B,1
2,CN104608125B,2
3,CN105234938B,3
4,CN106239505B,4
...,...,...
46530,CN111094095B,46530
46531,CN107818311B,46531
46532,CN109979593B,46532
46533,CN108471946B,46533


In [60]:
merge_df1 = pd.merge(g_vertice,patent_age,on='patent')
for name, age in zip(merge_df1['id'],merge_df1['pre_age']):
    g.vs.find(name)['age'] = age
sum_age = sum(merge_df1['pre_age'])   #  所有专利时间权重的加和

In [61]:
d,N = 0.5, g.vcount()
w_p = {p.index:1/N for p in g.vs}
for i in range(100000):
    diff = 0 # 初始化偏差
    tmp_dict = {p.index:0 for p in g.vs}
    for node in list(tmp_dict.keys()):
        tmp = 0
        parents_nodes = list(set([i.source for i in g.vs[node].in_edges()]))  # 指向node的父辈节点
        for j in parents_nodes:
            tmp += w_p[j] * g.vs[j]['age']/g.vs[j].outdegree()
        w_pki = (1-d) * g.vs[node]['age'] / sum_age + d*tmp
        tmp_dict[node] = w_pki
    telp = (1-sum(tmp_dict.values())) / N
    w_p_old = w_p.copy()
    for node in w_p.keys():
        w_p[node] = tmp_dict[node] + telp
        diff += abs(w_p[node] - w_p_old[node])
        
    print(i,diff)   
    if diff < 0.00000001:
        break

0 0.2136527907143084
1 0.029732342350130204
2 0.0035004154190986007
3 0.00040231797447933063
4 4.030194706260983e-05
5 3.7940551085451987e-06
6 3.6090559963557953e-07
7 3.254923302648448e-08
8 3.091521912981313e-09


In [62]:
pr_2 = [{"Patent":p[0]['name'], 'TAPR':p[1]} for p in zip(g.vs, w_p.values())]
vertices_df = pd.DataFrame(pr_2) 
vertices_df.sort_values('TAPR',ascending=False,ignore_index=True,inplace=True)
vertices_df

,Patent,TAPR
0,CN103824054B,0.000196
1,CN106250812B,0.000187
2,CN107154023B,0.000176
3,CN103530609B,0.000169
4,CN106204449B,0.000162
...,...,...
46530,CN1119765C,0.000009
46531,CN1074849C,0.000008
46532,CN1035290C,0.000007
46533,CN1023353C,0.000007


In [63]:
# 专利权人信息， 专利信息， 包括节点的id和名称
assignees_nodes = []
assignees_names = []
patents_nodes = []
patents_names = []
for i in h_g.vs:
    if 'CN' == i['name'][:2]:
        patents_names.append(i['name'])
        patents_nodes.append(i.index)
    else:
        assignees_nodes.append(i.index)
        assignees_names.append(i['name'])

# 专利列表
patents_df = pd.DataFrame()
patents_df['Patent'] = patents_names
patents_df['nodes'] = patents_nodes
# 专利权人列表
assignees_df = pd.DataFrame()
assignees_df['node'] = assignees_names
assignees_df['nodes'] = assignees_nodes

In [64]:
patents_merge = pd.merge(vertices_df,patents_df,on='Patent')

N_a, N_p, d = len(assignees_nodes), len(patents_nodes), 0.5
W_p = dict(zip(patents_merge['nodes'],patents_merge['TAPR']))
W_a = {i:1/N_a for i in assignees_nodes}
# 获得专利权人对应的发明专利的所有集合
W_ca = {}
for i in assignees_nodes:    
    W_ca[i] = [j.source for j in h_g.vs[i].all_edges()]
# 获得专利对应的所有专利权人集合
W_cp = {}
for i in patents_nodes:
    W_cp[i] = [j.target for j in h_g.vs[i].all_edges()]
    
W_p0,W_a0 = W_p.copy(), W_a.copy()

for i in range(10000):  
    diff = 0 # 初始化偏差
    W_a_1 = W_a.copy()
    W_p_1 = W_p.copy()
    for key in list(W_a.keys()):
        tmp = 0
        for paper in W_ca[key]:
            tmp += W_p_1[paper]/len(W_cp[paper]) 
        W_a[key] = (1 - d)*W_a0[key] + d * tmp
        
    tel_1 = (1 - sum(W_a.values()))/len(W_a)
    for key in list(W_a.keys()):
        W_a[key] += tel_1
        diff += abs(W_a[key] - W_a_1[key])
    
    for key in list(W_p.keys()):  # 遍历专利数据
        tmp = 0
        for author in W_cp[key]:
            tmp += W_a_1[author]/len(W_ca[author])
        W_p[key] = (1 - d)*W_p0[key] + d * tmp
        
    tel_2 = (1 - sum(W_p.values()))/N_p
    for key in list(W_p.keys()):
        W_p[key] += tel_2
        diff += abs(W_p[key] - W_p_1[key])
    
    print(i, 'diff: ', diff)
    if diff < 0.00000001:
        break


0 diff:  1.2808492653055181
1 diff:  0.6178854652074511
2 diff:  0.3011767775505996
3 diff:  0.14851005275938048
4 diff:  0.07354091443390949
5 diff:  0.03655050400153637
6 diff:  0.018209234809687985
7 diff:  0.009079022876260112
8 diff:  0.004529292626424528
9 diff:  0.0022605441644154046
10 diff:  0.0011285154688285971
11 diff:  0.0005634655672158539
12 diff:  0.00028137335099963085
13 diff:  0.00014051825896101752
14 diff:  7.017901635810584e-05
15 diff:  3.50520584017008e-05
16 diff:  1.750798607140824e-05
17 diff:  8.745522050406846e-06
18 diff:  4.3686731715709175e-06
19 diff:  2.1823914089201097e-06
20 diff:  1.0902622825036455e-06
21 diff:  5.446913434150348e-07
22 diff:  2.721364196125636e-07
23 diff:  1.3596884681105683e-07
24 diff:  6.793889454157473e-08
25 diff:  3.3950302028008355e-08
26 diff:  1.696751109781748e-08
27 diff:  8.48043488837151e-09


In [65]:
test_df.columns=['node','citation_count','rank']

In [66]:
taprw = pd.DataFrame.from_dict(W_a,orient='index',columns=['TAPRW']).reset_index()
taprw.columns=['nodes','TAPRW']
result_merge = pd.merge(assignees_df,taprw,on='nodes')
result_merge.sort_values(by=['TAPRW'],ascending=[False],ignore_index=True, inplace=True)

merge_data3 = pd.merge(test_df,result_merge,on='node')
merge_data3['TAPRW_rank'] = merge_data3['TAPRW'].rank(axis=0,method='min',ascending=False)
# merge_data3.to_excel("./results/算法结果/TAPRW_rank.xlsx",index=None)

In [67]:
N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = merge_data3.iloc[:top_N]['TAPRW_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 
# merge_data3.iloc[:200].to_csv("Top_200_taprw.csv",index=None)

Top N count:  10 , 总数量：  7 ,Average： 0.7 , NDCG@K score:  0.732
Top N count:  20 , 总数量：  16 ,Average： 0.8 , NDCG@K score:  0.861
Top N count:  30 , 总数量：  23 ,Average： 0.767 , NDCG@K score:  0.833
Top N count:  40 , 总数量：  29 ,Average： 0.725 , NDCG@K score:  0.802
Top N count:  50 , 总数量：  36 ,Average： 0.72 , NDCG@K score:  0.79
Top N count:  60 , 总数量：  43 ,Average： 0.717 , NDCG@K score:  0.792
Top N count:  70 , 总数量：  50 ,Average： 0.714 , NDCG@K score:  0.785
Top N count:  80 , 总数量：  54 ,Average： 0.675 , NDCG@K score:  0.756
Top N count:  90 , 总数量：  61 ,Average： 0.678 , NDCG@K score:  0.755
Top N count:  100 , 总数量：  67 ,Average： 0.67 , NDCG@K score:  0.747
Top N count:  200 , 总数量：  135 ,Average： 0.675 , NDCG@K score:  0.737
Top N count:  300 , 总数量：  198 ,Average： 0.66 , NDCG@K score:  0.716
Top N count:  400 , 总数量：  272 ,Average： 0.68 , NDCG@K score:  0.729
Top N count:  500 , 总数量：  321 ,Average： 0.642 , NDCG@K score:  0.693
Top N count:  1000 , 总数量：  650 ,Average： 0.65 , NDCG@K score:  

# APR算法

In [96]:
h_edges_1 = pd.read_excel("./results/最终数据/train_data.xlsx",sheet_name=0)

h_edges_2 = pd.read_excel("./results/最终数据/train_data.xlsx",sheet_name=1)

edges_3 = pd.DataFrame()
edges_3['citing'] = h_edges_2['cited']
edges_3['cited'] = h_edges_2['citing']

concat_1 = pd.concat([h_edges_2,edges_3])
concat_2 = pd.concat([concat_1,h_edges_1]).reset_index()

citing = concat_2['citing'].tolist()
cited = concat_2['cited'].tolist()
h_edges = []
for p in zip(citing,cited):
    h_edges.append((p[0],p[1]))

In [69]:
g = Graph.TupleList(h_edges,vertex_name_attr='name',edge_attrs=None,directed=True)

In [70]:
vertices_df = pd.DataFrame([{"node":p[0]['name'], 'PR':p[1]} for p in zip(g.vs, g.pagerank(damping=0.5))])
apr_merge = pd.merge(test_df, vertices_df, on='node')

apr_merge['APR_rank'] = apr_merge['PR'].rank(axis=0,method='min',ascending=False)
apr_merge.to_excel("./results/算法结果/APR_rank.xlsx",index=None)


N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = apr_merge.iloc[:top_N]['APR_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 
# apr_merge.iloc[:200].to_csv("Top_200_apr.csv",index=None)

Top N count:  10 , 总数量：  7 ,Average： 0.7 , NDCG@K score:  0.732
Top N count:  20 , 总数量：  15 ,Average： 0.75 , NDCG@K score:  0.828
Top N count:  30 , 总数量：  20 ,Average： 0.667 , NDCG@K score:  0.76
Top N count:  40 , 总数量：  24 ,Average： 0.6 , NDCG@K score:  0.707
Top N count:  50 , 总数量：  31 ,Average： 0.62 , NDCG@K score:  0.714
Top N count:  60 , 总数量：  38 ,Average： 0.633 , NDCG@K score:  0.726
Top N count:  70 , 总数量：  44 ,Average： 0.629 , NDCG@K score:  0.718
Top N count:  80 , 总数量：  49 ,Average： 0.613 , NDCG@K score:  0.702
Top N count:  90 , 总数量：  55 ,Average： 0.611 , NDCG@K score:  0.698
Top N count:  100 , 总数量：  61 ,Average： 0.61 , NDCG@K score:  0.695
Top N count:  200 , 总数量：  125 ,Average： 0.625 , NDCG@K score:  0.694
Top N count:  300 , 总数量：  185 ,Average： 0.617 , NDCG@K score:  0.678
Top N count:  400 , 总数量：  243 ,Average： 0.608 , NDCG@K score:  0.664
Top N count:  500 , 总数量：  299 ,Average： 0.598 , NDCG@K score:  0.653
Top N count:  1000 , 总数量：  589 ,Average： 0.589 , NDCG@K score:

# APN算法

In [71]:
assignees_nodes = []  
assignees_names = []
patents_nodes = []
patents_names = []
for i in g.vs:
    if 'CN' == i['name'][:2]:
        patents_names.append(i['name'])
        patents_nodes.append(i.index)
        i['type'] = 'p'
    else:
        assignees_nodes.append(i.index)
        assignees_names.append(i['name'])
        i['type'] = 'a'

# 专利列表
patents_df = pd.DataFrame()
patents_df['Patent'] = patents_names
patents_df['nodes'] = patents_nodes

# 专利权人列表
assignees_df = pd.DataFrame()
assignees_df['node'] = assignees_names
assignees_df['nodes'] = assignees_nodes



In [72]:
N_a, N_p, d = len(assignees_nodes), len(patents_nodes), 0.5
N_all = N_a + N_p

W_p = {i:1/N_all for i in patents_nodes} # 初始化
W_a = {i:1/N_all for i in assignees_nodes}

# 获得专利权人所对应的发明专利的所有集合
W_ca = {}
for i in assignees_nodes:
    W_ca[i] = [j.target for j in g.vs[i].out_edges()]   # 所有专利的数据

# 获得专利对应的参考专利和专利权人数量
W_cb, W_cc, W_cp = {}, {}, {}  # 被引专利, 参考专利 以及  专利权人
for i in patents_nodes:
    W_cb[i] = [j.source for j in g.vs[i].in_edges() if 'CN' in g.vs[j.source]['name'][:2]]
    W_cc[i] = [j.target for j in g.vs[i].out_edges() if 'CN' in g.vs[j.target]['name'][:2]]
    W_cp[i] = [j.target for j in g.vs[i].out_edges() if 'CN' not in g.vs[j.target]['name'][:2]]


W_p0, W_a0 = W_p.copy(), W_a.copy()
for i in range(10000):
    diff = 0
    W_a_1 = W_a.copy()
    W_p_1 = W_p.copy()
   
    for assignee in list(W_a.keys()):
        tmp = 0 
        for patent in W_ca[assignee]:
            tmp += W_p_1[patent]/len(W_cp[patent])
        W_a[assignee] = (1-d)/N_all + d * tmp
    # 专利
    for patent in list(W_p.keys()):
        tmp = 0
        for assignee in W_cp[patent]:   # 遍历专利权人的影响
            tmp += W_a_1[assignee] / len(W_ca[assignee])
        
        for reference in W_cb[patent]:     # 遍历专利的父辈节点
            tmp +=  W_p_1[reference] / len(W_cc[reference])  # 除以父辈节点的输出节点
        
        W_p[patent] = (1-d)/N_all + d * tmp
    
    telep = (1-sum(W_p.values())-sum(W_a.values())) / N_all   # teleport
    for key in list(W_p.keys()):
        W_p[key] += telep
        diff += abs(W_p[key] - W_p_1[key])
    for key in list(W_a.keys()):
        W_a[key] += telep
        diff += abs(W_a[key] - W_a_1[key])

    print(i, 'diff: ', diff)
    if diff < 0.00000001:
        break

0 diff:  0.7886942081177114
1 diff:  0.3747537133157698
2 diff:  0.1926669168424891
3 diff:  0.09291368802345287
4 diff:  0.048513460233884614
5 diff:  0.02431715335005309
6 diff:  0.012824750776256945
7 diff:  0.006642574030980539
8 diff:  0.0035160618038488347
9 diff:  0.0018662136701033008
10 diff:  0.0009878615673052091
11 diff:  0.0005322304805345378
12 diff:  0.0002837262876300394
13 diff:  0.00015361812790808758
14 diff:  8.25441173158729e-05
15 diff:  4.491498567984014e-05
16 diff:  2.434006616425352e-05
17 diff:  1.3299816604769918e-05
18 diff:  7.27193587554052e-06
19 diff:  3.995131864530068e-06
20 diff:  2.210983891643311e-06
21 diff:  1.2284419048255263e-06
22 diff:  6.8610283806036e-07
23 diff:  3.8521388521813936e-07
24 diff:  2.1752937554894043e-07
25 diff:  1.2351457086248778e-07
26 diff:  7.061877864142835e-08
27 diff:  4.063421347473613e-08
28 diff:  2.3527813343283667e-08
29 diff:  1.3726561824159322e-08
30 diff:  8.044298702927376e-09


In [73]:
# vertices_df = pd.DataFrame([{"node":p[0]['name'], 'APN':p[1]} for p in zip(g.vs, W_a.values())])

apn1 = pd.DataFrame.from_dict(W_a,orient='index',columns=['APN']).reset_index()
apn1.columns=['nodes','APN']
result_merge1 = pd.merge(assignees_df,apn1,on='nodes')

apr_merge = pd.merge(test_df, result_merge1, on='node')
apr_merge['APN_rank'] = apr_merge['APN'].rank(axis=0,method='min',ascending=False)

apr_merge.to_excel("./results/算法结果/APN_rank.xlsx",index=None)

N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = apr_merge.iloc[:top_N]['APN_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 
# apr_merge.iloc[:200].to_csv("Top_200_apn.csv",index=None)

Top N count:  10 , 总数量：  6 ,Average： 0.6 , NDCG@K score:  0.653
Top N count:  20 , 总数量：  16 ,Average： 0.8 , NDCG@K score:  0.861
Top N count:  30 , 总数量：  22 ,Average： 0.733 , NDCG@K score:  0.81
Top N count:  40 , 总数量：  28 ,Average： 0.7 , NDCG@K score:  0.784
Top N count:  50 , 总数量：  34 ,Average： 0.68 , NDCG@K score:  0.76
Top N count:  60 , 总数量：  39 ,Average： 0.65 , NDCG@K score:  0.739
Top N count:  70 , 总数量：  45 ,Average： 0.643 , NDCG@K score:  0.729
Top N count:  80 , 总数量：  48 ,Average： 0.6 , NDCG@K score:  0.691
Top N count:  90 , 总数量：  58 ,Average： 0.644 , NDCG@K score:  0.724
Top N count:  100 , 总数量：  65 ,Average： 0.65 , NDCG@K score:  0.729
Top N count:  200 , 总数量：  126 ,Average： 0.63 , NDCG@K score:  0.699
Top N count:  300 , 总数量：  193 ,Average： 0.643 , NDCG@K score:  0.702
Top N count:  400 , 总数量：  252 ,Average： 0.63 , NDCG@K score:  0.685
Top N count:  500 , 总数量：  309 ,Average： 0.618 , NDCG@K score:  0.672
Top N count:  1000 , 总数量：  616 ,Average： 0.616 , NDCG@K score:  0.662

# 带权重的APN算法

In [74]:
assignees_nodes = []  
assignees_names = []
patents_nodes = []
patents_names = []
for i in g.vs:
    if 'CN' == i['name'][:2]:
        patents_names.append(i['name'])
        patents_nodes.append(i.index)
        i['type'] = 'p'
    else:
        assignees_nodes.append(i.index)
        assignees_names.append(i['name'])
        i['type'] = 'a'

# 专利列表
patents_df = pd.DataFrame()
patents_df['Patent'] = patents_names
patents_df['nodes'] = patents_nodes

# 专利权人列表
assignees_df = pd.DataFrame()
assignees_df['node'] = assignees_names
assignees_df['nodes'] = assignees_nodes

In [75]:
N_a, N_p, d, beta = len(assignees_nodes), len(patents_nodes), 0.5, 0.3
N_all = N_a + N_p

W_p = {i:1/N_all for i in patents_nodes} # 初始化
W_a = {i:1/N_all for i in assignees_nodes}

# 获得专利权人所对应的发明专利的所有集合
W_ca = {}
for i in assignees_nodes:
    W_ca[i] = [j.target for j in g.vs[i].out_edges()]   # 所有专利的数据

# 获得专利对应的参考专利和专利权人数量
W_cb, W_cc, W_cp = {}, {}, {}  # 被引专利, 参考专利 以及  专利权人
for i in patents_nodes:
    W_cb[i] = [j.source for j in g.vs[i].in_edges() if 'CN' in g.vs[j.source]['name'][:2]]
    W_cc[i] = [j.target for j in g.vs[i].out_edges() if 'CN' in g.vs[j.target]['name'][:2]]
    W_cp[i] = [j.target for j in g.vs[i].out_edges() if 'CN' not in g.vs[j.target]['name'][:2]]


W_p0, W_a0 = W_p.copy(), W_a.copy()
for i in range(10000):
    diff = 0
    W_a_1 = W_a.copy()
    W_p_1 = W_p.copy()
   
    for assignee in list(W_a.keys()):
        tmp = 0 
        for patent in W_ca[assignee]:
            tmp += W_p_1[patent]/len(W_cp[patent])
        W_a[assignee] = (1-d)/N_all + d * (1 - beta) * tmp
    # 专利
    for patent in list(W_p.keys()):
        tmp = 0
        for assignee in W_cp[patent]:   # 遍历专利权人的影响
            tmp += W_a_1[assignee] / len(W_ca[assignee])
        
        for reference in W_cb[patent]:     # 遍历专利的父辈节点
            tmp += beta * W_p_1[reference] / len(W_cc[reference])  # 除以父辈节点的输出节点
        
        W_p[patent] = (1-d)/N_all + d * tmp
    
    telep = (1-sum(W_p.values())-sum(W_a.values())) / N_all   # teleport
    for key in list(W_p.keys()):
        W_p[key] += telep
        diff += abs(W_p[key] - W_p_1[key])
    for key in list(W_a.keys()):
        W_a[key] += telep
        diff += abs(W_a[key] - W_a_1[key])

    print(i, 'diff: ', diff)
    if diff < 0.00000001:
        break

0 diff:  0.4987012959440975
1 diff:  0.18174464275830599
2 diff:  0.06785687498270648
3 diff:  0.025643155000189734
4 diff:  0.009821183886999109
5 diff:  0.0037933524642819786
6 diff:  0.001485689621428885
7 diff:  0.0005853561259056127
8 diff:  0.0002345980104019539
9 diff:  9.414658524951332e-05
10 diff:  3.853340921651633e-05
11 diff:  1.571685819225558e-05
12 diff:  6.542005035491614e-06
13 diff:  2.704364758662678e-06
14 diff:  1.1387264987213485e-06
15 diff:  4.7682467291419584e-07
16 diff:  2.0208237988402274e-07
17 diff:  8.514734993947046e-08
18 diff:  3.624125776459951e-08
19 diff:  1.535717607088943e-08
20 diff:  6.5521601844026355e-09


In [76]:
apn2 = pd.DataFrame.from_dict(W_p,orient='index',columns=['WAPN']).reset_index()
apn2.columns=['nodes','WAPN']

In [77]:
apn2

,nodes,WAPN
0,0,0.000012
1,2,0.000012
2,4,0.000011
3,6,0.000017
4,8,0.000012
...,...,...
46530,56738,0.000015
46531,56739,0.000015
46532,56740,0.000015
46533,56741,0.000018


In [78]:
result_merge2 = pd.merge(patents_df,apn2,on='nodes')
result_merge2

,Patent,nodes,WAPN
0,CN105844614B,0,0.000012
1,CN104608125B,2,0.000012
2,CN106239505B,4,0.000011
3,CN108972536B,6,0.000017
4,CN109154276B,8,0.000012
...,...,...,...
46530,CN107316011B,56738,0.000015
46531,CN111094095B,56739,0.000015
46532,CN107818311B,56740,0.000015
46533,CN108471946B,56741,0.000018


In [79]:
apn2 = pd.DataFrame.from_dict(W_a,orient='index',columns=['WAPN']).reset_index()
apn2.columns=['nodes','WAPN']
result_merge1 = pd.merge(assignees_df,apn2,on='nodes')
apr_merge = pd.merge(test_df, result_merge1, on='node')
apr_merge['WAPN_rank'] = apr_merge['WAPN'].rank(axis=0,method='min',ascending=False)

In [110]:
apr_merge[apr_merge['node']=='深圳硅基智能科技有限公司']

,node,citation_count,rank,nodes,WAPN,WAPN_rank
135,深圳硅基智能科技有限公司,46,135,35583,0.000029,1187.0


In [80]:
apn2 = pd.DataFrame.from_dict(W_a,orient='index',columns=['WAPN']).reset_index()
apn2.columns=['nodes','WAPN']
result_merge1 = pd.merge(assignees_df,apn2,on='nodes')
apr_merge = pd.merge(test_df, result_merge1, on='node')
apr_merge['WAPN_rank'] = apr_merge['WAPN'].rank(axis=0,method='min',ascending=False)

apr_merge.to_excel("./results/算法结果/WAPN_rank.xlsx",index=None)

N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = apr_merge.iloc[:top_N]['WAPN_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 
# apr_merge.iloc[:200].to_csv("Top_200_wapn.csv",index=None)

Top N count:  10 , 总数量：  7 ,Average： 0.7 , NDCG@K score:  0.732
Top N count:  20 , 总数量：  15 ,Average： 0.75 , NDCG@K score:  0.828
Top N count:  30 , 总数量：  22 ,Average： 0.733 , NDCG@K score:  0.809
Top N count:  40 , 总数量：  29 ,Average： 0.725 , NDCG@K score:  0.802
Top N count:  50 , 总数量：  36 ,Average： 0.72 , NDCG@K score:  0.79
Top N count:  60 , 总数量：  38 ,Average： 0.633 , NDCG@K score:  0.727
Top N count:  70 , 总数量：  47 ,Average： 0.671 , NDCG@K score:  0.753
Top N count:  80 , 总数量：  52 ,Average： 0.65 , NDCG@K score:  0.734
Top N count:  90 , 总数量：  56 ,Average： 0.622 , NDCG@K score:  0.71
Top N count:  100 , 总数量：  63 ,Average： 0.63 , NDCG@K score:  0.715
Top N count:  200 , 总数量：  130 ,Average： 0.65 , NDCG@K score:  0.716
Top N count:  300 , 总数量：  196 ,Average： 0.653 , NDCG@K score:  0.71
Top N count:  400 , 总数量：  260 ,Average： 0.65 , NDCG@K score:  0.702
Top N count:  500 , 总数量：  315 ,Average： 0.63 , NDCG@K score:  0.683
Top N count:  1000 , 总数量：  631 ,Average： 0.631 , NDCG@K score:  0.

# TAWAPN算法

In [81]:
patent_age = pd.read_excel("./results/最终数据/HIN_node.xlsx")
patent_age.columns = ['patent','granted_date','age','AI_keywords_count','AI_COUNT']

In [82]:

# 计算所有专利时间权重的加和
# patent_age['pre_age'] = 1/(patent_age['age']+1)  # 时间函数f(age_i) 采用倒数形式来表示
patent_age['pre_age'] = [math.exp(-0.1* i) for i in patent_age['age'].tolist()] # 采用 指数时间函数

id_mapping = dict((k['name'],k.index) for k in g.vs)
g_vertice = pd.DataFrame.from_dict(id_mapping,orient='index',columns=['id']).reset_index()
g_vertice.columns=['patent','id']
g_vertice

,patent,id
0,CN105844614B,0
1,广东工业大学,1
2,CN104608125B,2
3,精工爱普生株式会社,3
4,CN106239505B,4
...,...,...
56740,CN107818311B,56740
56741,CN108471946B,56741
56742,珀迪迈垂克斯公司,56742
56743,CN110073404B,56743


In [83]:
merge_df1 = pd.merge(g_vertice,patent_age,on='patent')
for name, age in zip(merge_df1['id'],merge_df1['pre_age']):
    g.vs.find(name)['age'] = age

In [84]:
sum_age = sum(merge_df1['pre_age'])   #  所有专利时间权重的加和
assignees_nodes = []  
assignees_names = []
patents_nodes = []
patents_names = []
for i in g.vs:
    if 'CN' == i['name'][:2]:
        patents_names.append(i['name'])
        patents_nodes.append(i.index)
        i['type'] = 'p'
    else:
        assignees_nodes.append(i.index)
        assignees_names.append(i['name'])
        i['type'] = 'a'

# 专利列表
patents_df = pd.DataFrame()
patents_df['Patent'] = patents_names
patents_df['nodes'] = patents_nodes

# 专利权人列表
assignees_df = pd.DataFrame()
assignees_df['node'] = assignees_names
assignees_df['nodes'] = assignees_nodes

In [85]:
N_a, N_p, d, beta = len(assignees_nodes), len(patents_nodes), 0.5, 0.3
N_all = N_a + N_p

W_p = {i:1/N_p for i in patents_nodes} # 初始化
W_a = {i:1/N_a for i in assignees_nodes}

# 获得专利权人所对应的发明专利的所有集合
W_ca = {}
for i in assignees_nodes:
    W_ca[i] = [j.target for j in g.vs[i].out_edges()]   # 所有专利的数据

# 获得专利对应的参考专利和专利权人数量
W_cb, W_cc, W_cp = {}, {}, {}  # 被引专利, 参考专利 以及  专利权人
for i in patents_nodes:
    W_cb[i] = [j.source for j in g.vs[i].in_edges() if 'CN' in g.vs[j.source]['name'][:2]]
    W_cc[i] = [j.target for j in g.vs[i].out_edges() if 'CN' in g.vs[j.target]['name'][:2]]
    W_cp[i] = [j.target for j in g.vs[i].out_edges() if 'CN' not in g.vs[j.target]['name'][:2]]


W_p0, W_a0 = W_p.copy(), W_a.copy()
for i in range(10000):
    diff = 0
    W_a_1 = W_a.copy()
    W_p_1 = W_p.copy()
   
    for assignee in list(W_a.keys()):
        tmp = 0 
        for patent in W_ca[assignee]:
            tmp += W_p_1[patent]/len(W_cp[patent])
        W_a[assignee] = (1-d) * W_a0[assignee] + d * (1 - beta) * tmp
    # 专利
    for patent in list(W_p.keys()):
        tmp = 0
        for assignee in W_cp[patent]:   # 遍历专利权人的影响
            tmp += W_a_1[assignee] / len(W_ca[assignee])
        
        for reference in W_cb[patent]:     # 遍历专利的父辈节点
            tmp += beta * g.vs[reference]['age'] *  W_p_1[reference] / len(W_cc[reference])  # 除以父辈节点的输出节点
        
        W_p[patent] = (1-d) * g.vs[patent]['age'] / sum_age + d * tmp
    
    
    telep_1 = (1-sum(W_p.values())) / N_p   # teleport
    for key in list(W_p.keys()):
        W_p[key] += telep_1
        diff += abs(W_p[key] - W_p_1[key])
    
    telep_2 = (1-sum(W_a.values()))/ N_a
    for key in list(W_a.keys()):
        W_a[key] += telep_2
        diff += abs(W_a[key] - W_a_1[key])
        
#     telep = (1-sum(W_p.values())-sum(W_a.values())) / N_all   # teleport
#     for key in list(W_p.keys()):
#         W_p[key] += telep
#         diff += abs(W_p[key] - W_p_1[key])
#     for key in list(W_a.keys()):
#         W_a[key] += telep
#         diff += abs(W_a[key] - W_a_1[key])

    print(i, 'diff: ', diff)
    if diff < 0.00000001:
        break

0 diff:  1.0880144921020853
1 diff:  0.42714733634113977
2 diff:  0.17495955401462981
3 diff:  0.0704325385345365
4 diff:  0.029410063760657152
5 diff:  0.011996715005481455
6 diff:  0.005058507488174858
7 diff:  0.002081038295531388
8 diff:  0.0008806249576587636
9 diff:  0.00036431328037914477
10 diff:  0.00015461390985870135
11 diff:  6.42259880566337e-05
12 diff:  2.7315715175922205e-05
13 diff:  1.1392766423634566e-05
14 diff:  4.848842757587491e-06
15 diff:  2.030701553452531e-06
16 diff:  8.64320956713118e-07
17 diff:  3.6361082830005094e-07
18 diff:  1.5476334029163067e-07
19 diff:  6.542936068815247e-08
20 diff:  2.7876324984961107e-08
21 diff:  1.1821365442679255e-08
22 diff:  5.038595085900453e-09


In [86]:
tawapn = pd.DataFrame.from_dict(W_a,orient='index',columns=['TAWAPN']).reset_index()
tawapn.columns=['nodes','TAWAPN']
result_merge3 = pd.merge(assignees_df,tawapn,on='nodes')
tawapr_merge = pd.merge(test_df, result_merge3, on='node')
tawapr_merge['TAWAPN_rank'] = tawapr_merge['TAWAPN'].rank(axis=0,method='min',ascending=False)

tawapr_merge.to_excel("./results/算法结果/TAWAPN_rank.xlsx",index=None)

N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = tawapr_merge.iloc[:top_N]['TAWAPN_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 
# tawapr_merge.iloc[:200].to_csv("Top_200_tawapn.csv",index=None)


Top N count:  10 , 总数量：  6 ,Average： 0.6 , NDCG@K score:  0.646
Top N count:  20 , 总数量：  14 ,Average： 0.7 , NDCG@K score:  0.793
Top N count:  30 , 总数量：  24 ,Average： 0.8 , NDCG@K score:  0.858
Top N count:  40 , 总数量：  30 ,Average： 0.75 , NDCG@K score:  0.819
Top N count:  50 , 总数量：  37 ,Average： 0.74 , NDCG@K score:  0.807
Top N count:  60 , 总数量：  42 ,Average： 0.7 , NDCG@K score:  0.779
Top N count:  70 , 总数量：  51 ,Average： 0.729 , NDCG@K score:  0.799
Top N count:  80 , 总数量：  56 ,Average： 0.7 , NDCG@K score:  0.777
Top N count:  90 , 总数量：  63 ,Average： 0.7 , NDCG@K score:  0.773
Top N count:  100 , 总数量：  69 ,Average： 0.69 , NDCG@K score:  0.762
Top N count:  200 , 总数量：  136 ,Average： 0.68 , NDCG@K score:  0.741
Top N count:  300 , 总数量：  200 ,Average： 0.667 , NDCG@K score:  0.723
Top N count:  400 , 总数量：  271 ,Average： 0.677 , NDCG@K score:  0.726
Top N count:  500 , 总数量：  327 ,Average： 0.654 , NDCG@K score:  0.705
Top N count:  1000 , 总数量：  649 ,Average： 0.649 , NDCG@K score:  0.692


# TAWHAPN算法
在TAWAPN的基础上，加了专利权人的创新类型得分，如果是高校或者科研院所设置为3，企业设置为2，个人设置为1；加入专利权人位置信誉得分，为位置的倒数/总的得分

In [87]:
h_edges_2['order_score'] = round(1/h_edges_2['order'],3)   # 获得专利权人所在的顺序

new_df =pd.DataFrame() # 构建新的DataFrame
new_df['citing'] = h_edges_2.groupby("citing").sum().reset_index()['citing']  # 专利名称
new_df['sum_order'] = h_edges_2.groupby("citing").sum().reset_index()['order_score']  # 总的排名得分
new_merge_df = pd.merge(h_edges_2,new_df,on='citing')
new_merge_df

,citing,cited,Assignee_type,Assignee_score,order,year,order_score,sum_order
0,CN105844614B,广东工业大学,U,3,1,2019,1.0,1.0
1,CN104608125B,精工爱普生株式会社,E,2,1,2019,1.0,1.0
2,CN106239505B,广东电网有限责任公司电力科学研究院,R,3,1,2018,1.0,1.0
3,CN108972536B,西门子（中国）有限公司,E,2,1,2021,1.0,1.0
4,CN109154276B,维斯塔斯风力系统集团公司,E,2,1,2020,1.0,1.0
...,...,...,...,...,...,...,...,...
51192,CN107316011B,杭州励飞软件技术有限公司,E,2,1,2021,1.0,1.0
51193,CN111094095B,动态AD有限责任公司,E,2,1,2021,1.0,1.0
51194,CN107818311B,豪威科技（北京）股份有限公司,E,2,1,2021,1.0,1.0
51195,CN108471946B,珀迪迈垂克斯公司,E,2,1,2023,1.0,1.0


In [88]:
name_index = []  # 从图网络中获得专利的名称及其索引值
for i in g.vs:
    name_index.append({"citing":i['name'],"citing_index":i.index})
name_index_df = pd.DataFrame(name_index)
name_merge1 = pd.merge(new_merge_df,name_index_df,on='citing')
# 从图网络中获得专利权人的名称及其索引
name_index_2 = []
for i in g.vs:
    name_index_2.append({"cited":i['name'],"cited_index":i.index})
name_index_df2 = pd.DataFrame(name_index_2)
name_merge2 = pd.merge(name_merge1,name_index_df2,on='cited')

name_merge2['order_value'] = round(name_merge2['order_score'] / name_merge2['sum_order'],2) # 获得专利权人所在专利的顺序获得的信誉分配值

In [89]:
# 构建一个字典，用于存放专利权人：专利：信誉分配值， 主要用于后面进行查找和分配信誉分配值
order_dict = {}
for index in name_merge2.index:
    row = name_merge2.iloc[index]
    row_key = row['cited_index']
    row_sub_key = row['citing_index']
    row_value = row['order_value']
    if row_key not in order_dict:
        order_dict[row_key] = {}
    order_dict[row_key][row_sub_key] = row_value

In [90]:
# 获得专利权人的类型得分，将其认为不同类型的专利权人其创新程度值的贡献不一样
assignee_df = pd.DataFrame()
assignee_df['node'] = h_edges_2['cited']
assignee_df['Assignee_score'] = h_edges_2['Assignee_score']
assignee_df = assignee_df.drop_duplicates()  # 删除重复项
sum_score = sum(assignee_df['Assignee_score'])  # 获得所有专利权人的类型得分之和
# 将获得的专利权人类型得分写入网络图中
for name, count in zip(assignee_df['node'], assignee_df['Assignee_score']):
    g.vs.find(name)['Assignee_score'] = count

In [91]:
# 先初始化各参数，并进行遍历迭代
N_a, N_p, d, beta = len(assignees_nodes), len(patents_nodes), 0.7, 0.4  # 包括阻尼系数，包括权重系数  需要修改

W_p = {i:1/N_p for i in patents_nodes} # 初始化专利及其得分
W_a = {i:1/N_a for i in assignees_nodes}  # 初始化专利权人及其得分

# 获得专利权人所对应的发明专利的所有集合
W_ca = {}
for i in assignees_nodes:
    W_ca[i] = [j.target for j in g.vs[i].out_edges()]   # 所有专利的数据

# 获得专利对应的参考专利和专利权人数量
W_cb, W_cc, W_cp = {}, {}, {}  # 被引专利, 参考专利 以及  专利权人
for i in patents_nodes:
    W_cb[i] = [j.source for j in g.vs[i].in_edges() if 'CN' in g.vs[j.source]['name'][:2]]
    W_cc[i] = [j.target for j in g.vs[i].out_edges() if 'CN' in g.vs[j.target]['name'][:2]]
    W_cp[i] = [j.target for j in g.vs[i].out_edges() if 'CN' not in g.vs[j.target]['name'][:2]]

# 初始化
W_p0, W_a0 = W_p.copy(), W_a.copy()
for i in range(10000):
    diff = 0
    W_a_1 = W_a.copy()
    W_p_1 = W_p.copy()
    # 遍历专利权人节点
    for assignee in list(W_a.keys()):
        tmp = 0 
        for patent in W_ca[assignee]:
            # 将专利权人所在专利的信誉分配度赋值给对应的专利，说明排名越靠前的专利权人其重要程度越大
            tmp += W_p_1[patent] * order_dict[assignee][patent]         # /len(W_cp[patent])
        
        assignee_value = g.vs[assignee]['Assignee_score']   # 专利权人类型得分，我们设定为高校和科研机构为3分，企业为2分，个人为1分
        W_a[assignee] = (1-d)  * assignee_value / sum_score  + d * (1 - beta) * tmp  # * W_a0[assignee]
    
    # 遍历专利节点
    for patent in list(W_p.keys()):
        tmp = 0
        for assignee in W_cp[patent]:   # 遍历专利权人的影响
            tmp += W_a_1[assignee]  / len(W_ca[assignee])  # 专利权人的影响力除以其所写论文的数量  W_ca代表assignee所写的论文总数量
        
        for reference in W_cb[patent]:     # 遍历专利的父辈节点
            tmp += beta * g.vs[reference]['age']  *   W_p_1[reference] / len(W_cc[reference])  # 除以父辈节点的输出节点
        
        W_p[patent] = (1-d) * g.vs[patent]['age'] / sum_age  + d * tmp  # 更新专利的得分，包含专利的年龄占比，以及其从网络中获得的得分    
    
    telep_1 = (1-sum(W_p.values())) / N_p   # teleport
    for key in list(W_p.keys()):
        W_p[key] += telep_1
        diff += abs(W_p[key] - W_p_1[key])
    
    telep_2 = (1-sum(W_a.values()))/ N_a
    for key in list(W_a.keys()):
        W_a[key] += telep_2
        diff += abs(W_a[key] - W_a_1[key])

    print(i, 'diff: ', diff)
    if diff < 0.00000001:
        break

0 diff:  1.4345082588302616
1 diff:  0.7194505464660648
2 diff:  0.38349054954171885
3 diff:  0.19906849115804118
4 diff:  0.10876744639618514
5 diff:  0.05754659987086231
6 diff:  0.031712758095960025
7 diff:  0.01697226793037598
8 diff:  0.009394543759557908
9 diff:  0.005073213497812483
10 diff:  0.002814028312029161
11 diff:  0.001532003381411062
12 diff:  0.0008500907415445605
13 diff:  0.00046677974841183723
14 diff:  0.0002593352971827798
15 diff:  0.0001433375538836341
16 diff:  7.966296136594219e-05
17 diff:  4.4250527864569734e-05
18 diff:  2.4590690037071407e-05
19 diff:  1.3707873444958158e-05
20 diff:  7.617329439022491e-06
21 diff:  4.256413649484016e-06
22 diff:  2.365121454522972e-06
23 diff:  1.324416000144035e-06
24 diff:  7.360504397851385e-07
25 diff:  4.128502190407582e-07
26 diff:  2.295198130892191e-07
27 diff:  1.2892182092063217e-07
28 diff:  7.17095933529138e-08
29 diff:  4.03407436869191e-08
30 diff:  2.2452067885118592e-08
31 diff:  1.264568322881357e-08
32 

In [92]:
tawapn = pd.DataFrame.from_dict(W_a,orient='index',columns=['FWAPN']).reset_index()
tawapn.columns=['nodes','FWAPN']
result_merge3 = pd.merge(assignees_df,tawapn,on='nodes')
tawapr_merge = pd.merge(test_df, result_merge3, on='node')
tawapr_merge['FWAPN_rank'] = tawapr_merge['FWAPN'].rank(axis=0,method='min',ascending=False)

tawapr_merge.to_excel("./results/算法结果/FWAPN_rank.xlsx",index=None)

N_list = [10,20,30,40,50,60,70,80,90,100,200,300,400,500,1000,2000]
for top_N in N_list:
    sub_data = tawapr_merge.iloc[:top_N]['FWAPN_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 

Top N count:  10 , 总数量：  7 ,Average： 0.7 , NDCG@K score:  0.732
Top N count:  20 , 总数量：  15 ,Average： 0.75 , NDCG@K score:  0.826
Top N count:  30 , 总数量：  24 ,Average： 0.8 , NDCG@K score:  0.858
Top N count:  40 , 总数量：  31 ,Average： 0.775 , NDCG@K score:  0.837
Top N count:  50 , 总数量：  39 ,Average： 0.78 , NDCG@K score:  0.84
Top N count:  60 , 总数量：  44 ,Average： 0.733 , NDCG@K score:  0.803
Top N count:  70 , 总数量：  52 ,Average： 0.743 , NDCG@K score:  0.809
Top N count:  80 , 总数量：  57 ,Average： 0.713 , NDCG@K score:  0.784
Top N count:  90 , 总数量：  63 ,Average： 0.7 , NDCG@K score:  0.772
Top N count:  100 , 总数量：  68 ,Average： 0.68 , NDCG@K score:  0.755
Top N count:  200 , 总数量：  138 ,Average： 0.69 , NDCG@K score:  0.749
Top N count:  300 , 总数量：  200 ,Average： 0.667 , NDCG@K score:  0.721
Top N count:  400 , 总数量：  273 ,Average： 0.682 , NDCG@K score:  0.73
Top N count:  500 , 总数量：  333 ,Average： 0.666 , NDCG@K score:  0.713
Top N count:  1000 , 总数量：  644 ,Average： 0.644 , NDCG@K score:  0.

# AIHIN算法
## 将《管理世界》所获得的AI关键词作为专利的指标，加到算法中


In [112]:
patent_ai = pd.read_excel("./results/最终数据/HIN_node.xlsx")
patent_ai.columns = ['patent','granted_date','age','AI_keywords_count','AI_COUNT']


id_mapping = dict((k['name'],k.index) for k in g.vs)
g_vertice = pd.DataFrame.from_dict(id_mapping,orient='index',columns=['id']).reset_index()
g_vertice.columns=['patent','id']

merge_df1 = pd.merge(g_vertice,patent_ai,on='patent')
for name, ai in zip(merge_df1['id'],merge_df1['AI_keywords_count']):
    g.vs.find(name)['AI_keywords_count'] = ai

In [115]:
# 先初始化各参数，并进行遍历迭代
N_a, N_p, d, beta = len(assignees_nodes), len(patents_nodes), 0.7, 0.4  # 包括阻尼系数，包括权重系数  需要修改

W_p = {i:1/N_p for i in patents_nodes} # 初始化专利及其得分
W_a = {i:1/N_a for i in assignees_nodes}  # 初始化专利权人及其得分

# 获得专利权人所对应的发明专利的所有集合
W_ca = {}
for i in assignees_nodes:
    W_ca[i] = [j.target for j in g.vs[i].out_edges()]   # 所有专利的数据

# 获得专利对应的参考专利和专利权人数量
W_cb, W_cc, W_cp = {}, {}, {}  # 被引专利, 参考专利 以及  专利权人
for i in patents_nodes:
    W_cb[i] = [j.source for j in g.vs[i].in_edges() if 'CN' in g.vs[j.source]['name'][:2]]
    W_cc[i] = [j.target for j in g.vs[i].out_edges() if 'CN' in g.vs[j.target]['name'][:2]]
    W_cp[i] = [j.target for j in g.vs[i].out_edges() if 'CN' not in g.vs[j.target]['name'][:2]]
sum_ai = sum(merge_df1['AI_keywords_count'])

# 初始化
W_p0, W_a0 = W_p.copy(), W_a.copy()
for i in range(10000):
    diff = 0
    W_a_1 = W_a.copy()
    W_p_1 = W_p.copy()
    # 遍历专利权人节点
    for assignee in list(W_a.keys()):
        tmp = 0 
        for patent in W_ca[assignee]:
            # 将专利权人所在专利的信誉分配度赋值给对应的专利，说明排名越靠前的专利权人其重要程度越大
            tmp += W_p_1[patent] * order_dict[assignee][patent]         # /len(W_cp[patent])
        
        assignee_value = g.vs[assignee]['Assignee_score']   # 专利权人类型得分，我们设定为高校和科研机构为3分，企业为2分，个人为1分
        W_a[assignee] = (1-d)  * assignee_value / sum_score  + d * (1 - beta) * tmp  # * W_a0[assignee]
    
    # 遍历专利节点
    for patent in list(W_p.keys()):
        tmp = 0
        for assignee in W_cp[patent]:   # 遍历专利权人的影响
            tmp += W_a_1[assignee]  / len(W_ca[assignee])  # 专利权人的影响力除以其所写论文的数量  W_ca代表assignee所写的论文总数量
        
        for reference in W_cb[patent]:     # 遍历专利的父辈节点
            tmp += beta * g.vs[reference]['age'] * g.vs[reference]['AI_keywords_count'] *   W_p_1[reference] / len(W_cc[reference])  # 除以父辈节点的输出节点
        
        W_p[patent] = (1-d) * (0.5 * g.vs[patent]['age'] / sum_age + 0.5*g.vs[patent]['AI_keywords_count'] / sum_ai) + d * tmp  # 更新专利的得分，包含专利的年龄占比，以及其从网络中获得的得分    
        # W_p[patent] = (1-d) * (g.vs[patent]['AI_keywords_count'] / sum_ai ) + d * tmp  # 更新专利的得分，包含专利的年龄占比，以及其从网络中获得的得分    

    
    telep_1 = (1-sum(W_p.values())) / N_p   # teleport
    for key in list(W_p.keys()):
        W_p[key] += telep_1
        diff += abs(W_p[key] - W_p_1[key])
    
    telep_2 = (1-sum(W_a.values()))/ N_a
    for key in list(W_a.keys()):
        W_a[key] += telep_2
        diff += abs(W_a[key] - W_a_1[key])

    print(i, 'diff: ', diff)
    if diff < 0.00000001:
        break

0 diff:  1.5835934482491145
1 diff:  0.7894426417336451
2 diff:  0.5074524714289023
3 diff:  0.26240646801100215
4 diff:  0.1660815750281335
5 diff:  0.09337216533858367
6 diff:  0.05807976506444955
7 diff:  0.035089904206177844
8 diff:  0.022123320114151898
9 diff:  0.013813000087394004
10 diff:  0.008738962738378677
11 diff:  0.005582815513399221
12 diff:  0.0035317079953966057
13 diff:  0.00229284149933848
14 diff:  0.001466868326010533
15 diff:  0.0009608920591805918
16 diff:  0.0006239711304236978
17 diff:  0.0004101592448345692
18 diff:  0.00026786838331684473
19 diff:  0.00017651901357261193
20 diff:  0.000115867831323474
21 diff:  7.648369846182895e-05
22 diff:  5.043399542789339e-05
23 diff:  3.335010910367178e-05
24 diff:  2.2094407185214974e-05
25 diff:  1.4634739356320307e-05
26 diff:  9.735752250624196e-06
27 diff:  6.4775023611121995e-06
28 diff:  4.321547659089568e-06
29 diff:  2.8883943936939542e-06
30 diff:  1.9359764289349298e-06
31 diff:  1.297655980079e-06
32 diff: 

In [116]:
tawapn = pd.DataFrame.from_dict(W_a,orient='index',columns=['AIHIN']).reset_index()
tawapn.columns=['nodes','AIHIN']
result_merge3 = pd.merge(assignees_df,tawapn,on='nodes')
tawapr_merge = pd.merge(test_df, result_merge3, on='node')
tawapr_merge['AIHIN_rank'] = tawapr_merge['AIHIN'].rank(axis=0,method='min',ascending=False)

tawapr_merge.to_excel("./results/算法结果/AIHIN_rank.xlsx",index=None)

N_list = [10,30,80,150,200,500]
for top_N in N_list:
    sub_data = tawapr_merge.iloc[:top_N]['AIHIN_rank'].values.tolist()
    in_data = [i for i in sub_data if int(i) <= top_N]
    tmp,sum_tmp = 0,0
    for i in range(len(sub_data)):
        rel_i = 0
        if int(sub_data[i]) <= top_N:
            rel_i = 1
        tmp += (2**rel_i - 1) / (math.log2(i+2))
        sum_tmp += (2-1)/(math.log2(i+2))
    ndcg = round(tmp/sum_tmp, 3)
    print("Top N count: ", top_N, ', 总数量： ',len(in_data), ',Average：', round(len(in_data)/top_N,3), ", NDCG@K score: ", ndcg) 

Top N count:  10 , 总数量：  5 ,Average： 0.5 , NDCG@K score:  0.62
Top N count:  30 , 总数量：  18 ,Average： 0.6 , NDCG@K score:  0.678
Top N count:  80 , 总数量：  50 ,Average： 0.625 , NDCG@K score:  0.686
Top N count:  150 , 总数量：  86 ,Average： 0.573 , NDCG@K score:  0.657
Top N count:  200 , 总数量：  114 ,Average： 0.57 , NDCG@K score:  0.646
Top N count:  500 , 总数量：  250 ,Average： 0.5 , NDCG@K score:  0.559


Top N count:  10 , 总数量：  6 ,Average： 0.6 , NDCG@K score:  0.697
Top N count:  30 , 总数量：  22 ,Average： 0.733 , NDCG@K score:  0.808
Top N count:  80 , 总数量：  62 ,Average： 0.775 , NDCG@K score:  0.831
Top N count:  150 , 总数量：  105 ,Average： 0.7 , NDCG@K score:  0.762
Top N count:  200 , 总数量：  133 ,Average： 0.665 , NDCG@K score:  0.726
Top N count:  500 , 总数量：  326 ,Average： 0.652 , NDCG@K score:  0.7K score:  0.702